# `uq_analysis_v2`: config-driven UQ analysis

This notebook is dataset-agnostic. Its behavior is controlled by a YAML config.

Configs created for this repo:
- `configs/uq_analysis_vw.yaml`
- `configs/uq_analysis_craigs.yaml`

Phase 2 (current): resolve run directories + check required files exist.


In [ ]:
# Thesis artifacts export (for LaTeX).
from pathlib import Path

EXPORT_ROOT = Path("thesis_artifacts")
OVERWRITE = True
EXPORT_TAG = None  # e.g. "craigslist_test" (optional disambiguation)

FIG_EXT = "pdf"  # "pdf" or "png"
FIG_ALSO_PNG = True  # when FIG_EXT="pdf", optionally emit .png too
FIG_DPI = 150
FIG_BBOX = "tight"



In [ ]:
# Setup + config loading.
from __future__ import annotations

from pathlib import Path
import sys
from typing import Any, Dict, Optional

import pandas as pd
import yaml

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

# Resolve project root (works from repo root or notebooks/)
NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "scripts").is_dir() and (NOTEBOOK_DIR / "configs").is_dir():
    PROJECT_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "scripts").is_dir() and (NOTEBOOK_DIR.parent / "configs").is_dir():
    PROJECT_ROOT = NOTEBOOK_DIR.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Thesis artifacts exporter.
from scripts.thesis_artifacts import ThesisArtifactsExporter

EXPORTER = ThesisArtifactsExporter(
    export_root=(PROJECT_ROOT / EXPORT_ROOT).resolve(),
    overwrite=bool(OVERWRITE),
    export_tag=(str(EXPORT_TAG) if EXPORT_TAG else None),
)
EXPORTER.prepare()
print("EXPORT_ROOT:", EXPORTER.export_root)


def savefig(
    fig,
    topic: str,
    *,
    method: str | None = None,
    dataset: str | None = None,
    split: str | None = None,
    ext: str | None = None,
    also_png: bool | None = None,
    dpi: int | None = None,
    bbox_inches: str | None = None,
    facecolor: str = "white",
    tag: str | None = None,
):
    ext = FIG_EXT if ext is None else ext
    also_png = FIG_ALSO_PNG if also_png is None else also_png
    dpi = FIG_DPI if dpi is None else dpi
    bbox_inches = FIG_BBOX if bbox_inches is None else bbox_inches
    return EXPORTER.savefig(
        fig,
        topic,
        method=method,
        dataset=dataset,
        split=split,
        ext=ext,
        also_png=also_png,
        dpi=dpi,
        bbox_inches=bbox_inches,
        facecolor=facecolor,
        tag=tag,
    )

def _resolve_path(p: Optional[str]) -> Optional[Path]:
    if not p:
        return None
    path = Path(str(p))
    return path if path.is_absolute() else (PROJECT_ROOT / path).resolve()


def _normalize_prefix(p: Optional[str]) -> str:
    if not p:
        return ""
    p = str(p)
    return p if p.startswith("_") else "_" + p


# Pick which dataset config to run.
# CONFIG_PATH = PROJECT_ROOT / "configs" / "uq_analysis_vw.yaml"
CONFIG_PATH = PROJECT_ROOT / "configs" / "uq_analysis_craigs.yaml"

cfg: Dict[str, Any] = yaml.safe_load(CONFIG_PATH.read_text()) or {}
settings = cfg.get("settings", {}) or {}

DATASET_KEY = settings.get("dataset_key")
EVAL_ROOT = _resolve_path(settings.get("eval_root")) or (PROJECT_ROOT / "outputs" / "evals")
RUN_PREFIX = _normalize_prefix(settings.get("run_prefix"))

OOD_LABELS = list(cfg.get("ood_labels", []) or [])
LARGE_ERROR = cfg.get("large_error", {}) or {}
COHORTS = cfg.get("cohorts", {}) or {}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("CONFIG_PATH :", CONFIG_PATH)
print("DATASET_KEY :", DATASET_KEY)
print("EVAL_ROOT   :", EVAL_ROOT)
print("RUN_PREFIX  :", RUN_PREFIX)
print("OOD_LABELS  :", OOD_LABELS)
print("LARGE_ERROR :", LARGE_ERROR)


In [ ]:
# Phase 2A: resolve run directories from config.
from typing import List

run_overrides = cfg.get("runs", {}) or {}
run_patterns = cfg.get("run_patterns", {}) or {}


def _pick_latest_dir(glob_pat: str) -> Optional[Path]:
    matches = sorted(EVAL_ROOT.glob(glob_pat))
    return matches[-1] if matches else None


def _resolve_run(pattern_key: str) -> Optional[Path]:
    explicit = run_overrides.get(pattern_key)
    if explicit:
        p = _resolve_path(str(explicit))
        return p if p and p.exists() else None

    pat = run_patterns.get(pattern_key)
    if not pat:
        return None
    pat = str(pat).format(prefix=RUN_PREFIX)
    return _pick_latest_dir(pat)


# Map method keys to pattern keys in the config
RUN_DIRS: Dict[str, Optional[Path]] = {
    "point": _resolve_run("point"),
    "lpl": _resolve_run("laplace_mc"),
    "gau": _resolve_run("gauss_mc"),
    "esm_lpl": _resolve_run("laplace_ensemble"),
    "esm_gau": _resolve_run("gauss_ensemble"),
    "nf_lpl": _resolve_run("nf_laplace"),
    "nf_gau": _resolve_run("nf_gauss"),
    "dido_lpl": _resolve_run("dido_laplace"),
    "dido_gau": _resolve_run("dido_gauss"),
}

df_runs = pd.DataFrame([
    {
        "method": k,
        "run_dir": str(v) if v else None,
        "exists": bool(v and v.exists()),
    }
    for k, v in RUN_DIRS.items()
])
print(df_runs)


In [ ]:
# Phase 2B: check expected files exist for each resolved run dir.
from typing import Iterable


def _first_found(run_dir: Path, candidates: Iterable[str]) -> Optional[str]:
    for name in candidates:
        if (run_dir / name).exists():
            return name
    return None


METHOD_KIND = {
    "point": "regression",
    "lpl": "regression",
    "gau": "regression",
    "esm_lpl": "ensemble",
    "esm_gau": "ensemble",
    "nf_lpl": "nf",
    "nf_gau": "nf",
    "dido_lpl": "dido",
    "dido_gau": "dido",
}

SPLIT_CANDIDATES = {
    "train": ["train_preds.csv", "preds_train.csv", "mc_preds_train.csv"],
    "val": ["val_preds.csv", "preds_val.csv", "mc_preds_val.csv"],
    "test": ["test_preds.csv", "preds_test.csv", "mc_preds_test.csv"],
}

rows = []
for method, run_dir in RUN_DIRS.items():
    if not run_dir:
        continue
    kind = METHOD_KIND.get(method, "unknown")

    # ID split files
    if kind == "regression":
        for split, cands in SPLIT_CANDIDATES.items():
            rows.append({
                "method": method,
                "kind": kind,
                "scope": "id",
                "label": split,
                "file": _first_found(run_dir, cands),
            })
        for label in OOD_LABELS:
            rows.append({
                "method": method,
                "kind": kind,
                "scope": "ood",
                "label": label,
                "file": _first_found(run_dir, [f"mc_preds_ood_{label}.csv", f"ood_{label}_test.csv"]),
            })

    elif kind == "ensemble":
        for split in ("train", "val", "test"):
            rows.append({
                "method": method,
                "kind": kind,
                "scope": "id",
                "label": split,
                "file": _first_found(run_dir, [f"ensemble_preds_{split}.csv"]),
            })
        for label in OOD_LABELS:
            rows.append({
                "method": method,
                "kind": kind,
                "scope": "ood",
                "label": label,
                "file": _first_found(run_dir, [f"ensemble_preds_ood_{label}.csv"]),
            })

    elif kind == "nf":
        for split in ("train", "val", "test"):
            rows.append({
                "method": method,
                "kind": kind,
                "scope": "id",
                "label": split,
                "file": _first_found(run_dir, [f"flow_eval_{split}.csv"]),
            })
        for label in OOD_LABELS:
            rows.append({
                "method": method,
                "kind": kind,
                "scope": "ood",
                "label": label,
                "file": _first_found(run_dir, [f"flow_eval_ood_{label}.csv"]),
            })

    elif kind == "dido":
        for split in ("train", "val", "test"):
            rows.append({
                "method": method,
                "kind": kind,
                "scope": "id",
                "label": split,
                "file": _first_found(run_dir, [f"dido_{split}.csv"]),
            })
        for label in OOD_LABELS:
            rows.append({
                "method": method,
                "kind": kind,
                "scope": "ood",
                "label": label,
                "file": _first_found(run_dir, [f"dido_ood_{label}.csv"]),
            })

df_files = pd.DataFrame(rows)
df_files["exists"] = df_files["file"].notna()
print(df_files)

# Cohort dataset sanity
dataset_csv = _resolve_path((COHORTS.get("dataset_csv") if isinstance(COHORTS, dict) else None))
print("\n[cohorts]")
print(" enabled    :", bool(COHORTS.get("enabled", False)) if isinstance(COHORTS, dict) else False)
print(" dataset_csv:", dataset_csv)
if dataset_csv:
    print(" exists     :", dataset_csv.exists())


## Phase 3: dataset-agnostic loaders (file-first)

Goal: load ID/OOD artifacts for each enabled method into a small, standardized schema without loading large MC sample columns.


In [ ]:
from dataclasses import dataclass
from typing import List, Optional


@dataclass(frozen=True)
class MethodSpec:
    key: str
    label: str
    head_type: str
    aleatoric_source: str
    epistemic_source: str
    kind: str  # regression | ensemble | nf | dido
    run_dir_key: str
    variant: Optional[str] = None  # point | analytic | mc (for regression)


enable = (cfg.get("enable_methods", {}) or {}) if isinstance(cfg, dict) else {}
def _enabled(name: str, default: bool = True) -> bool:
    return bool(enable.get(name, default))


METHODS: List[MethodSpec] = []

if _enabled("point"):
    METHODS.append(MethodSpec("point", "Point", "point", "none", "none", "regression", "point", "point"))

if _enabled("laplace_analytic"):
    METHODS.append(MethodSpec("lpl_analytic", "Laplace (analytic)", "laplace", "analytic", "none", "regression", "lpl", "analytic"))
if _enabled("laplace_mc"):
    METHODS.append(MethodSpec("lpl_mc", "Laplace + MC", "laplace", "analytic", "mc", "regression", "lpl", "mc"))
if _enabled("laplace_ensemble"):
    METHODS.append(MethodSpec("esm_lpl", "Laplace + ensemble", "laplace", "analytic", "ensemble", "ensemble", "esm_lpl"))

if _enabled("gauss_analytic"):
    METHODS.append(MethodSpec("gau_analytic", "Gauss (analytic)", "gauss", "analytic", "none", "regression", "gau", "analytic"))
if _enabled("gauss_mc"):
    METHODS.append(MethodSpec("gau_mc", "Gauss + MC", "gauss", "analytic", "mc", "regression", "gau", "mc"))
if _enabled("gauss_ensemble"):
    METHODS.append(MethodSpec("esm_gau", "Gauss + ensemble", "gauss", "analytic", "ensemble", "ensemble", "esm_gau"))

if _enabled("nf_laplace"):
    METHODS.append(MethodSpec("nf_lpl", "Laplace (NF)", "laplace", "nf", "none", "nf", "nf_lpl"))
if _enabled("nf_gauss"):
    METHODS.append(MethodSpec("nf_gau", "Gauss (NF)", "gauss", "nf", "none", "nf", "nf_gau"))

if _enabled("dido_laplace"):
    METHODS.append(MethodSpec("dido_lpl", "Laplace + DIDO", "laplace", "analytic", "dido", "dido", "dido_lpl"))
if _enabled("dido_gauss"):
    METHODS.append(MethodSpec("dido_gau", "Gauss + DIDO", "gauss", "analytic", "dido", "dido", "dido_gau"))


def _run_dir_for(spec: MethodSpec) -> Optional[Path]:
    p = RUN_DIRS.get(spec.run_dir_key)
    return p if isinstance(p, Path) and p.exists() else None


df_methods = pd.DataFrame([
    {
        "key": m.key,
        "label": m.label,
        "kind": m.kind,
        "variant": m.variant,
        "run_dir": str(_run_dir_for(m)) if _run_dir_for(m) else None,
    }
    for m in METHODS
])
print(df_methods)


In [ ]:
# File-first loaders (read only required columns).
import numpy as np
import pandas as pd


def _read_csv_usecols(path: Path, usecols: List[str]) -> pd.DataFrame:
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    keep = [c for c in usecols if c in cols]
    return pd.read_csv(path, usecols=keep)


def _first_existing(base: Path, names: List[str]) -> Path:
    for name in names:
        p = base / name
        if p.exists():
            return p
    return base / names[0]


def load_id_regression(run_dir: Path, split: str, *, variant: str) -> pd.DataFrame:
    """variant: point | analytic | mc"""
    run_dir = Path(run_dir)
    split = str(split).lower()

    # For historical reasons, test split is often named test_preds.csv
    if split == "test":
        split_candidates = ["test_preds.csv", "preds_test.csv"]
    else:
        split_candidates = [f"preds_{split}.csv", f"{split}_preds.csv"]

    if variant == "mc":
        path = _first_existing(run_dir, [f"mc_preds_{split}.csv", *split_candidates])
        df = _read_csv_usecols(
            path,
            [
                "id",
                "split",
                "head_type",
                "y_true",
                "y_pred_mc_mean",
                "y_pred_det",
                "sigma_ale_orig",
                "sigma_ale_raw",
                "sigma_epi_orig",
                "sigma_epi_raw",
            ],
        )
        if "y_pred_mc_mean" in df.columns:
            df = df.rename(columns={"y_pred_mc_mean": "mu"})
        elif "y_pred_det" in df.columns:
            df = df.rename(columns={"y_pred_det": "mu"})
        else:
            mc_cols = [c for c in df.columns if c.startswith("y_pred_mc_")]
            df["mu"] = df[mc_cols].mean(axis=1) if mc_cols else np.nan

        if "sigma_ale_raw" not in df.columns and "sigma_ale_orig" in df.columns:
            df = df.rename(columns={"sigma_ale_orig": "sigma_ale_raw"})
        if "sigma_epi_raw" not in df.columns and "sigma_epi_orig" in df.columns:
            df = df.rename(columns={"sigma_epi_orig": "sigma_epi_raw"})
        if "sigma_ale_raw" not in df.columns:
            df["sigma_ale_raw"] = np.nan
        if "sigma_epi_raw" not in df.columns:
            df["sigma_epi_raw"] = np.nan
        return df

    # point/analytic: prefer deterministic preds
    path = _first_existing(run_dir, split_candidates + [f"mc_preds_{split}.csv"])
    df = _read_csv_usecols(
        path,
        [
            "id",
            "split",
            "head_type",
            "y_true",
            "y_pred",
            "y_pred_det",
            "y_pred_mc_mean",
            "sigma_ale_raw",
            "sigma_ale_orig",
            "sigma_epi_raw",
            "sigma_epi_orig",
        ],
    )

    if "y_pred" in df.columns:
        df = df.rename(columns={"y_pred": "mu"})
    elif "y_pred_det" in df.columns:
        df = df.rename(columns={"y_pred_det": "mu"})
    elif "y_pred_mc_mean" in df.columns:
        df = df.rename(columns={"y_pred_mc_mean": "mu"})
    else:
        df["mu"] = np.nan

    if "sigma_ale_raw" not in df.columns and "sigma_ale_orig" in df.columns:
        df = df.rename(columns={"sigma_ale_orig": "sigma_ale_raw"})
    if "sigma_epi_raw" not in df.columns and "sigma_epi_orig" in df.columns:
        df = df.rename(columns={"sigma_epi_orig": "sigma_epi_raw"})

    # Point/analytic variants do not use epistemic here
    if variant in {"point", "analytic"}:
        df["sigma_epi_raw"] = np.nan

    if "sigma_ale_raw" not in df.columns:
        df["sigma_ale_raw"] = np.nan
    if "sigma_epi_raw" not in df.columns:
        df["sigma_epi_raw"] = np.nan
    return df


def load_id_ensemble(run_dir: Path, split: str) -> pd.DataFrame:
    path = Path(run_dir) / f"ensemble_preds_{split}.csv"
    df = _read_csv_usecols(
        path,
        [
            "id",
            "split",
            "head_type",
            "y_true",
            "y_pred_ens_mean",
            "sigma_ale_ens",
            "sigma_epi_ens",
            "y_pred_mean",
            "y_pred_method",
            "sigma_ale_orig",
            "sigma_epi_orig",
        ],
    )
    if "y_pred_ens_mean" in df.columns:
        df = df.rename(columns={"y_pred_ens_mean": "mu"})
    elif "y_pred_mean" in df.columns:
        df = df.rename(columns={"y_pred_mean": "mu"})
    elif "y_pred_method" in df.columns:
        df = df.rename(columns={"y_pred_method": "mu"})
    else:
        df["mu"] = np.nan
    if "sigma_ale_ens" in df.columns:
        df = df.rename(columns={"sigma_ale_ens": "sigma_ale_raw"})
    elif "sigma_ale_orig" in df.columns:
        df = df.rename(columns={"sigma_ale_orig": "sigma_ale_raw"})
    if "sigma_epi_ens" in df.columns:
        df = df.rename(columns={"sigma_epi_ens": "sigma_epi_raw"})
    elif "sigma_epi_orig" in df.columns:
        df = df.rename(columns={"sigma_epi_orig": "sigma_epi_raw"})
    if "sigma_ale_raw" not in df.columns:
        df["sigma_ale_raw"] = np.nan
    if "sigma_epi_raw" not in df.columns:
        df["sigma_epi_raw"] = np.nan
    return df


def load_id_nf(run_dir: Path, split: str) -> pd.DataFrame:
    split = str(split).lower()
    path = Path(run_dir) / f"flow_eval_{split}.csv"
    df = _read_csv_usecols(
        path,
        ["id", "split", "head_type", "y_true_orig", "y_pred_base_orig", "sigma_base_orig", "sigma_nf_orig"],
    )
    df = df.rename(columns={"y_true_orig": "y_true", "y_pred_base_orig": "mu"})
    if "sigma_nf_orig" in df.columns:
        df["sigma_ale_raw"] = df["sigma_nf_orig"]
    elif "sigma_base_orig" in df.columns:
        df["sigma_ale_raw"] = df["sigma_base_orig"]
    else:
        df["sigma_ale_raw"] = np.nan
    df["sigma_epi_raw"] = 0.0
    return df


def load_id_dido(run_dir: Path) -> pd.DataFrame:
    path = Path(run_dir) / "dido_test.csv"
    return _read_csv_usecols(path, ["id", "split", "head_type", "dido_strength_raw", "dido_vacuity_raw", "dido_entropy_raw"])


def load_ood_regression(run_dir: Path, label: str, *, variant: Optional[str] = None) -> pd.DataFrame:
    run_dir = Path(run_dir)
    label = str(label)
    if variant == "mc":
        path = _first_existing(run_dir, [f"mc_preds_ood_{label}.csv", f"ood_{label}_test.csv"])
    else:
        path = run_dir / f"ood_{label}_test.csv"
    df = _read_csv_usecols(
        path,
        [
            "id",
            "split",
            "head_type",
            "y_true",
            "y_pred_det",
            "y_pred_mc_mean",
            "y_pred",
            "sigma_ale_orig",
            "sigma_ale_raw",
            "sigma_epi_orig",
            "sigma_epi_raw",
        ],
    )
    if "y_pred_mc_mean" in df.columns:
        df = df.rename(columns={"y_pred_mc_mean": "mu"})
    elif "y_pred_det" in df.columns:
        df = df.rename(columns={"y_pred_det": "mu"})
    elif "y_pred" in df.columns:
        df = df.rename(columns={"y_pred": "mu"})
    else:
        df["mu"] = np.nan
    if "sigma_ale_raw" not in df.columns and "sigma_ale_orig" in df.columns:
        df = df.rename(columns={"sigma_ale_orig": "sigma_ale_raw"})
    if "sigma_epi_raw" not in df.columns and "sigma_epi_orig" in df.columns:
        df = df.rename(columns={"sigma_epi_orig": "sigma_epi_raw"})
    if "sigma_ale_raw" not in df.columns:
        df["sigma_ale_raw"] = np.nan
    if "sigma_epi_raw" not in df.columns:
        df["sigma_epi_raw"] = np.nan
    return df


def load_ood_nf(run_dir: Path, label: str) -> pd.DataFrame:
    path = Path(run_dir) / f"flow_eval_ood_{label}.csv"
    df = _read_csv_usecols(
        path,
        ["id", "split", "head_type", "y_true_orig", "y_pred_base_orig", "sigma_base_orig", "sigma_nf_orig"],
    )
    df = df.rename(columns={"y_true_orig": "y_true", "y_pred_base_orig": "mu"})
    if "sigma_nf_orig" in df.columns:
        df["sigma_ale_raw"] = df["sigma_nf_orig"]
    elif "sigma_base_orig" in df.columns:
        df["sigma_ale_raw"] = df["sigma_base_orig"]
    else:
        df["sigma_ale_raw"] = np.nan
    df["sigma_epi_raw"] = 0.0
    return df


def load_ood_dido(run_dir: Path, label: str) -> pd.DataFrame:
    path = Path(run_dir) / f"dido_ood_{label}.csv"
    return _read_csv_usecols(path, ["id", "split", "head_type", "dido_strength_raw", "dido_vacuity_raw", "dido_entropy_raw"])


In [ ]:
# Phase 3 verification: smoke-load ID:test and one OOD label per method.


def method_id_test_frame(spec: MethodSpec) -> pd.DataFrame:
    run_dir = _run_dir_for(spec)
    if run_dir is None:
        raise FileNotFoundError(f"No run_dir for {spec.key}")
    if spec.kind == "regression":
        return load_id_regression(run_dir, "test", variant=str(spec.variant))
    if spec.kind == "ensemble":
        return load_id_ensemble(run_dir, "test")
    if spec.kind == "nf":
        return load_id_nf(run_dir, "test")
    if spec.kind == "dido":
        return load_id_dido(run_dir)
    raise ValueError(f"Unknown kind={spec.kind}")


def method_ood_frame(spec: MethodSpec, label: str) -> pd.DataFrame:
    run_dir = _run_dir_for(spec)
    if run_dir is None:
        raise FileNotFoundError(f"No run_dir for {spec.key}")
    if spec.kind == "regression":
        variant = "mc" if spec.variant == "mc" else None
        return load_ood_regression(run_dir, label, variant=variant)
    if spec.kind == "ensemble":
        path = run_dir / f"ensemble_preds_ood_{label}.csv"
        df = _read_csv_usecols(
            path,
            [
                "id",
                "split",
                "head_type",
                "y_true",
                "y_pred_ens_mean",
                "sigma_ale_ens",
                "sigma_epi_ens",
                "y_pred_mean",
                "sigma_ale_orig",
                "sigma_epi_orig",
            ],
        )
        if "y_pred_ens_mean" in df.columns:
            df = df.rename(columns={"y_pred_ens_mean": "mu"})
        elif "y_pred_mean" in df.columns:
            df = df.rename(columns={"y_pred_mean": "mu"})
        if "sigma_ale_ens" in df.columns:
            df = df.rename(columns={"sigma_ale_ens": "sigma_ale_raw"})
        if "sigma_epi_ens" in df.columns:
            df = df.rename(columns={"sigma_epi_ens": "sigma_epi_raw"})
        return df
    if spec.kind == "nf":
        return load_ood_nf(run_dir, label)
    if spec.kind == "dido":
        return load_ood_dido(run_dir, label)
    raise ValueError(f"Unknown kind={spec.kind}")


def large_error_tau() -> float:
    mode = str((LARGE_ERROR or {}).get("mode", "quantile")).lower()
    if mode == "fixed":
        return float(LARGE_ERROR.get("threshold"))
    if mode != "quantile":
        raise ValueError(f"Unsupported large_error.mode={mode!r}")
    q = float(LARGE_ERROR.get("quantile", 0.95))
    baseline = str(LARGE_ERROR.get("baseline_method", "point"))
    base_spec = next((m for m in METHODS if m.key == baseline), None)
    if base_spec is None:
        raise KeyError(f"large_error.baseline_method={baseline!r} not enabled/available")
    df = method_id_test_frame(base_spec)
    y = pd.to_numeric(df.get("y_true"), errors="coerce").to_numpy(float)
    mu = pd.to_numeric(df.get("mu"), errors="coerce").to_numpy(float)
    ae = np.abs(y - mu)
    return float(np.nanquantile(ae, q))


TAU = large_error_tau()
print(f"[large_error] tau={TAU:.6g} (mode={LARGE_ERROR.get('mode')})")
EXPORTER.export_number("large_error_tau", TAU, dataset=DATASET_KEY, split="test")

print("\nID:test smoke load")
for spec in METHODS:
    try:
        df = method_id_test_frame(spec)
        print(f"  {spec.key:10s} n={len(df):,} cols={len(df.columns)}")
    except Exception as e:
        print(f"  [skip] {spec.key:10s}: {type(e).__name__}: {e}")

if OOD_LABELS:
    label = OOD_LABELS[0]
    print(f"\nOOD smoke load (label={label!r})")
    for spec in METHODS:
        try:
            df = method_ood_frame(spec, label)
            print(f"  {spec.key:10s} n={len(df):,} cols={len(df.columns)}")
        except Exception as e:
            print(f"  [skip] {spec.key:10s}: {type(e).__name__}: {e}")


## Phase 4: core ID metrics + calibration

This phase computes core predictive metrics on **ID:test** and (optionally) fits a simple \"variance scaling\" calibration (alpha/beta) on **ID:val**.


In [ ]:
# Phase 4A: calibration helpers + (optional) fit on ID:val

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

from scripts.uq_eval import gaussian_nll

N_RELIABILITY_BINS = int(settings.get("n_reliability_bins", cfg.get("n_reliability_bins", 10)))

# Toggle calibration fitting (metrics below still work if you skip this).
DO_FIT_CALIBRATION = True
CALIB_SPLIT = "val"


def gaussian_crps_mean(y: np.ndarray, mu: np.ndarray, sigma: np.ndarray) -> float:
    y = np.asarray(y, float)
    mu = np.asarray(mu, float)
    sigma = np.asarray(sigma, float)
    mask = np.isfinite(y) & np.isfinite(mu) & np.isfinite(sigma) & (sigma > 0)
    if not np.any(mask):
        return float("nan")
    y = y[mask]
    mu = mu[mask]
    sigma = sigma[mask]
    z = (y - mu) / sigma
    crps = sigma * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / np.sqrt(np.pi))
    return float(np.mean(crps))


def method_id_frame(spec: MethodSpec, split: str) -> pd.DataFrame:
    """Load an ID split for a method spec (file-first)."""
    run_dir = _run_dir_for(spec)
    if run_dir is None:
        raise FileNotFoundError(f"No run_dir for {spec.key}")

    split = str(split).lower()
    if spec.kind == "regression":
        return load_id_regression(run_dir, split, variant=str(spec.variant))
    if spec.kind == "ensemble":
        return load_id_ensemble(run_dir, split)
    if spec.kind == "nf":
        return load_id_nf(run_dir, split)
    if spec.kind == "dido":
        if split != "test":
            raise FileNotFoundError("DIDO eval artifacts are test-only in this repo")
        return load_id_dido(run_dir)
    raise ValueError(f"Unknown kind={spec.kind}")


# Calibration maps (filled by this cell if enabled)
try:
    alpha
except NameError:
    alpha = {}
try:
    beta
except NameError:
    beta = {}


def calibrated_sigmas(*, head_type: str, ale_src: str, epi_src: str, sigma_ale_raw: np.ndarray, sigma_epi_raw: np.ndarray):
    """Return (sigma_ale_cal, sigma_epi_cal, sigma_total_cal) in original target units."""
    # Fallback: NF uses analytic alpha if no explicit NF alpha was fit.
    a = alpha.get((head_type, ale_src))
    if a is None and str(ale_src) == "nf":
        a = alpha.get((head_type, "analytic"))
    if a is None:
        a = 1.0
    a = float(a)

    b = float(beta.get((head_type, ale_src, epi_src), 1.0))
    if str(epi_src) in {"none", "dido"}:
        b = 0.0

    s_a2 = a * (np.nan_to_num(np.asarray(sigma_ale_raw, float), nan=0.0) ** 2)
    s_e2 = b * (np.nan_to_num(np.asarray(sigma_epi_raw, float), nan=0.0) ** 2)
    s_a = np.sqrt(np.maximum(s_a2, 0.0))
    s_e = np.sqrt(np.maximum(s_e2, 0.0))
    s_t = np.sqrt(np.maximum(s_a2 + s_e2, 0.0))
    return s_a, s_e, s_t


if DO_FIT_CALIBRATION:
    alpha = {}
    beta = {}

    CALIB_OBJECTIVE = str(settings.get("calib_objective", cfg.get("calib_objective", "moment"))).lower().strip()
    if CALIB_OBJECTIVE not in {"moment", "nll", "crps"}:
        raise ValueError(f"Unsupported calib_objective={CALIB_OBJECTIVE!r} (expected moment|nll|crps)")

    # Search grids (log-spaced). Used for CRPS alpha and for NLL/CRPS beta.
    ALPHA_GRID = np.exp(np.linspace(np.log(1e-3), np.log(1e3), 41))
    BETA_GRID = np.exp(np.linspace(np.log(1e-3), np.log(1e3), 41))

    dataset_label = str(settings.get("dataset_key", cfg.get("dataset_key", "unknown")))
    print(f"[calib] dataset={dataset_label} split={CALIB_SPLIT} objective={CALIB_OBJECTIVE}")

    def _fit_alpha_moment(y: np.ndarray, mu: np.ndarray, s_ale: np.ndarray) -> float:
        err2 = (y - mu) ** 2
        s2 = np.nan_to_num(s_ale, nan=0.0) ** 2
        m = np.isfinite(err2) & np.isfinite(s2) & (s2 > 0)
        if not np.any(m):
            return float("nan")
        return float(np.mean(err2[m]) / np.mean(s2[m]))

    def _fit_alpha_crps(y: np.ndarray, mu: np.ndarray, s_ale: np.ndarray) -> float:
        s = np.asarray(s_ale, float)
        m = np.isfinite(y) & np.isfinite(mu) & np.isfinite(s) & (s > 0)
        if not np.any(m):
            return float("nan")
        y0 = y[m]
        mu0 = mu[m]
        s0 = s[m]
        best_a = 1.0
        best = float("inf")
        for a in ALPHA_GRID:
            sigma = np.sqrt(a) * s0
            v = gaussian_crps_mean(y0, mu0, sigma)
            if np.isfinite(v) and v < best:
                best = float(v)
                best_a = float(a)
        return float(best_a)

    def _fit_beta_grid(y: np.ndarray, mu: np.ndarray, s_ale: np.ndarray, s_epi: np.ndarray, a: float) -> float:
        s_a = np.asarray(s_ale, float)
        s_e = np.asarray(s_epi, float)
        m = np.isfinite(y) & np.isfinite(mu) & np.isfinite(s_a) & np.isfinite(s_e) & (s_a > 0) & (s_e >= 0)
        if not np.any(m):
            return float("nan")
        y0 = y[m]
        mu0 = mu[m]
        s_a0 = s_a[m]
        s_e0 = s_e[m]

        s_a2 = a * (s_a0 ** 2)
        s_e2 = (s_e0 ** 2)

        best_b = 1.0
        best = float("inf")
        for b in BETA_GRID:
            sigma2 = s_a2 + b * s_e2
            sigma = np.sqrt(np.maximum(sigma2, 1e-8))
            if CALIB_OBJECTIVE == "nll":
                v = gaussian_nll(y0, mu0, sigma)
            else:  # crps
                v = gaussian_crps_mean(y0, mu0, sigma)
            if np.isfinite(v) and v < best:
                best = float(v)
                best_b = float(b)
        return float(best_b)

    # 1) Fit alpha using methods with no epistemic component (epi_src == 'none').
    for spec in METHODS:
        if spec.head_type == "point" or spec.kind == "dido":
            continue
        if spec.epistemic_source != "none":
            continue
        try:
            df_val = method_id_frame(spec, CALIB_SPLIT)
        except Exception as e:
            print(f"[skip calib/alpha] {spec.key}: {type(e).__name__}: {e}")
            continue
        if "y_true" not in df_val.columns or "mu" not in df_val.columns:
            print(f"[skip calib/alpha] {spec.key}: missing y_true/mu")
            continue

        y = pd.to_numeric(df_val["y_true"], errors="coerce").to_numpy(float)
        mu = pd.to_numeric(df_val["mu"], errors="coerce").to_numpy(float)
        s_a = pd.to_numeric(df_val.get("sigma_ale_raw"), errors="coerce").to_numpy(float)

        if CALIB_OBJECTIVE == "crps":
            a = _fit_alpha_crps(y, mu, s_a)
        else:
            # 'moment' and 'nll' use the same closed-form scale for alpha.
            a = _fit_alpha_moment(y, mu, s_a)

        if not np.isfinite(a):
            print(f"[skip calib/alpha] {spec.key}: no finite data")
            continue
        alpha[(spec.head_type, spec.aleatoric_source)] = float(a)
        print(f"[calib/alpha] {spec.label}: alpha={a:.4g} (n={int(np.isfinite(s_a).sum())})")

    # 2) Fit beta for methods with epistemic component (mc/ensemble).
    for spec in METHODS:
        if spec.head_type == "point" or spec.kind == "dido":
            continue
        if spec.epistemic_source not in {"mc", "ensemble"}:
            continue
        try:
            df_val = method_id_frame(spec, CALIB_SPLIT)
        except Exception as e:
            print(f"[skip calib/beta] {spec.key}: {type(e).__name__}: {e}")
            continue
        if "y_true" not in df_val.columns or "mu" not in df_val.columns:
            print(f"[skip calib/beta] {spec.key}: missing y_true/mu")
            continue

        y = pd.to_numeric(df_val["y_true"], errors="coerce").to_numpy(float)
        mu = pd.to_numeric(df_val["mu"], errors="coerce").to_numpy(float)
        s_a = pd.to_numeric(df_val.get("sigma_ale_raw"), errors="coerce").to_numpy(float)
        s_e = pd.to_numeric(df_val.get("sigma_epi_raw"), errors="coerce").fillna(0.0).to_numpy(float)

        a = float(alpha.get((spec.head_type, spec.aleatoric_source), 1.0))

        if CALIB_OBJECTIVE == "moment":
            # Moment-matching beta: fit the residual variance explained by epistemic.
            err2 = (y - mu) ** 2
            s_a2 = np.nan_to_num(s_a, nan=0.0) ** 2
            s_e2 = np.nan_to_num(s_e, nan=0.0) ** 2
            mask = np.isfinite(err2) & np.isfinite(s_a2) & np.isfinite(s_e2) & (s_e2 > 0)
            if not np.any(mask):
                print(f"[skip calib/beta] {spec.key}: no finite data")
                continue
            resid = err2 - a * s_a2
            num = float(np.dot(s_e2[mask], resid[mask]))
            den = float(np.dot(s_e2[mask], s_e2[mask]))
            b = (num / den) if den > 0 else 1.0
            b = float(np.clip(b, 1e-8, 1e8))
        else:
            b = _fit_beta_grid(y, mu, s_a, s_e, a)
            if not np.isfinite(b):
                print(f"[skip calib/beta] {spec.key}: no finite data")
                continue

        beta[(spec.head_type, spec.aleatoric_source, spec.epistemic_source)] = float(b)
        print(f"[calib/beta] {spec.label}: beta={b:.4g} (alpha={a:.4g})")

    print(f"[calib] done dataset={dataset_label} objective={CALIB_OBJECTIVE} alpha keys={len(alpha)} beta keys={len(beta)}")
else:
    print("[calib] skipped (DO_FIT_CALIBRATION=False)")

# --- summary table for calibration (all 8 combos) ---
if DO_FIT_CALIBRATION:
    rows = []
    for head_type in ["laplace", "gauss"]:
        a_analytic = alpha.get((head_type, "analytic"))
        for epi in ["mc", "ensemble"]:
            b = beta.get((head_type, "analytic", epi))
            rows.append({
                "ale_source": head_type,
                "epi_source": epi,
                "alpha": float(a_analytic) if a_analytic is not None else float("nan"),
                "beta": float(b) if b is not None else float("nan"),
            })

        a_nf = alpha.get((head_type, "nf"))
        rows.append({
            "ale_source": f"{head_type} (nf)",
            "epi_source": "N/A",
            "alpha": float(a_nf) if a_nf is not None else float("nan"),
            "beta": float("nan"),
        })

    df_calib = pd.DataFrame(rows, columns=["ale_source", "epi_source", "alpha", "beta"])
    EXPORTER.export_table(df_calib, "calibration_params", dataset=DATASET_KEY, split=CALIB_SPLIT)
    display(df_calib)


In [ ]:
# Phase 4B: core predictive metrics on ID:test

USE_CALIBRATED_SIGMAS = True

rows = []
for spec in METHODS:
    if spec.kind == "dido":
        continue
    try:
        df = method_id_frame(spec, "test")
    except Exception as e:
        print(f"[skip metrics] {spec.key}: {type(e).__name__}: {e}")
        continue
    if "y_true" not in df.columns or "mu" not in df.columns:
        print(f"[skip metrics] {spec.key}: missing y_true/mu")
        continue

    y = pd.to_numeric(df["y_true"], errors="coerce").to_numpy(float)
    mu = pd.to_numeric(df["mu"], errors="coerce").to_numpy(float)
    mae = float(np.nanmean(np.abs(y - mu)))
    rmse = float(np.sqrt(np.nanmean((y - mu) ** 2)))

    if spec.head_type == "point":
        rows.append({
            "method": spec.label,
            "key": spec.key,
            "kind": spec.kind,
            "head_type": spec.head_type,
            "aleatoric_source": spec.aleatoric_source,
            "epistemic_source": spec.epistemic_source,
            "n": int(len(df)),
            "MAE": mae,
            "RMSE": rmse,
            "NLL": np.nan,
            "CRPS": np.nan,
            "mean_sigma_total": np.nan,
        })
        continue

    s_a_raw = pd.to_numeric(df.get("sigma_ale_raw"), errors="coerce").to_numpy(float)
    s_e_raw = pd.to_numeric(df.get("sigma_epi_raw"), errors="coerce").fillna(0.0).to_numpy(float)
    if USE_CALIBRATED_SIGMAS:
        _, _, s_t = calibrated_sigmas(
            head_type=spec.head_type,
            ale_src=spec.aleatoric_source,
            epi_src=spec.epistemic_source,
            sigma_ale_raw=s_a_raw,
            sigma_epi_raw=s_e_raw,
        )
    else:
        s_t = np.sqrt(np.maximum(np.nan_to_num(s_a_raw, nan=0.0) ** 2 + np.nan_to_num(s_e_raw, nan=0.0) ** 2, 0.0))

    mask = np.isfinite(y) & np.isfinite(mu) & np.isfinite(s_t) & (s_t > 0)
    nll = float(gaussian_nll(y[mask], mu[mask], s_t[mask])) if np.any(mask) else float("nan")
    crps = gaussian_crps_mean(y, mu, s_t)

    rows.append({
        "method": spec.label,
        "key": spec.key,
        "kind": spec.kind,
        "head_type": spec.head_type,
        "aleatoric_source": spec.aleatoric_source,
        "epistemic_source": spec.epistemic_source,
        "n": int(len(df)),
        "MAE": mae,
        "RMSE": rmse,
        "NLL": nll,
        "CRPS": crps,
        "mean_sigma_total": float(np.nanmean(s_t)) if np.any(np.isfinite(s_t)) else float("nan"),
    })

df_core_metrics_id = pd.DataFrame(rows).sort_values(["MAE", "NLL"], na_position="last")
EXPORTER.export_table(df_core_metrics_id, "metrics_overall", dataset=DATASET_KEY, split="test")
display(df_core_metrics_id)


In [ ]:
# Phase 4C: calibration diagnostic (ID:test) - nominal vs empirical coverage

NOMINAL_LEVELS = list(range(10, 100, 10))


def coverage_at_levels(y: np.ndarray, mu: np.ndarray, sigma: np.ndarray, levels: list[int]) -> dict[int, float]:
    cov = {}
    y = np.asarray(y, float)
    mu = np.asarray(mu, float)
    sigma = np.asarray(sigma, float)
    base_mask = np.isfinite(y) & np.isfinite(mu) & np.isfinite(sigma) & (sigma > 0)
    for lvl in levels:
        z = float(norm.ppf(0.5 + (lvl / 200.0)))
        m = base_mask & np.isfinite(z)
        cov[lvl] = float(np.mean(np.abs(y[m] - mu[m]) <= z * sigma[m])) if np.any(m) else float("nan")
    return cov


rows = []
for spec in METHODS:
    if spec.head_type == "point" or spec.kind == "dido":
        continue
    try:
        df = method_id_frame(spec, "test")
    except Exception as e:
        print(f"[skip coverage] {spec.key}: {type(e).__name__}: {e}")
        continue
    if "y_true" not in df.columns or "mu" not in df.columns:
        continue

    y = pd.to_numeric(df["y_true"], errors="coerce").to_numpy(float)
    mu = pd.to_numeric(df["mu"], errors="coerce").to_numpy(float)
    s_a_raw = pd.to_numeric(df.get("sigma_ale_raw"), errors="coerce").to_numpy(float)
    s_e_raw = pd.to_numeric(df.get("sigma_epi_raw"), errors="coerce").fillna(0.0).to_numpy(float)
    s_raw = np.sqrt(np.maximum(np.nan_to_num(s_a_raw, nan=0.0) ** 2 + np.nan_to_num(s_e_raw, nan=0.0) ** 2, 0.0))
    _, _, s_cal = calibrated_sigmas(
        head_type=spec.head_type,
        ale_src=spec.aleatoric_source,
        epi_src=spec.epistemic_source,
        sigma_ale_raw=s_a_raw,
        sigma_epi_raw=s_e_raw,
    )

    cov_raw = coverage_at_levels(y, mu, s_raw, NOMINAL_LEVELS)
    cov_cal = coverage_at_levels(y, mu, s_cal, NOMINAL_LEVELS)
    ece = float(100.0 * np.nanmean([abs(cov_cal[l] - (l / 100.0)) for l in NOMINAL_LEVELS]))
    rows.append({
        "method": spec.label,
        "key": spec.key,
        "ece_pp": ece,
        **{f"cov_raw_{l}": cov_raw[l] for l in NOMINAL_LEVELS},
        **{f"cov_cal_{l}": cov_cal[l] for l in NOMINAL_LEVELS},
    })

df_cov = pd.DataFrame(rows).sort_values("ece_pp")
EXPORTER.export_table(df_cov, "calibration_coverage_full", dataset=DATASET_KEY, split="test")
EXPORTER.export_table(df_cov[["method", "ece_pp"] + [f"cov_cal_{l}" for l in NOMINAL_LEVELS]], "calibration_coverage", dataset=DATASET_KEY, split="test")
display(df_cov[["method", "ece_pp"] + [f"cov_cal_{l}" for l in NOMINAL_LEVELS]])

# Small multiples plot (raw vs calibrated)
methods_order = df_cov["method"].tolist()
n = len(methods_order)
if n:
    ncols = 3
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows), sharex=False, sharey=False)
    axes = np.array(axes).reshape(-1)
    for ax, method in zip(axes, methods_order):
        row = df_cov[df_cov["method"] == method].iloc[0]
        cov_raw = [100.0 * row[f"cov_raw_{l}"] for l in NOMINAL_LEVELS]
        cov_cal = [100.0 * row[f"cov_cal_{l}"] for l in NOMINAL_LEVELS]
        ax.plot(NOMINAL_LEVELS, cov_raw, "--", color="0.6", linewidth=2.0, label="raw")
        ax.plot(NOMINAL_LEVELS, cov_cal, "-o", color="tab:blue", label="cal")
        ax.plot(NOMINAL_LEVELS, NOMINAL_LEVELS, "k-", alpha=0.6, label="ideal")
        ax.set_title(f"{method}\nECE={row['ece_pp']:.2f} pp")
        ax.set_xlabel("Nominal (%)")
        ax.set_ylabel("Empirical (%)")
        ax.set_xlim(0, 100)
        ax.set_ylim(0, 100)
        ax.grid(True, alpha=0.2)
    for ax in axes[len(methods_order):]:
        ax.set_axis_off()
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="lower center", ncol=3, frameon=False)
    fig.suptitle("Total-uncertainty coverage on ID:test (raw vs calibrated)")
    fig.tight_layout(rect=[0, 0.06, 1, 0.98])
    savefig(fig, "calibration_coverage", dataset=DATASET_KEY, split="test")
    plt.show()


In [ ]:
# Phase 4D: binned error vs uncertainty (ID:test)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

BIN_METRIC = "mae"  # "mae" or "crps"
N_BINS = int(N_RELIABILITY_BINS) if "N_RELIABILITY_BINS" in globals() else 10
USE_CALIBRATED_SIGMAS_LOCAL = USE_CALIBRATED_SIGMAS if "USE_CALIBRATED_SIGMAS" in globals() else True


def _gaussian_crps_per_row(y: np.ndarray, mu: np.ndarray, sigma: np.ndarray) -> np.ndarray:
    y = np.asarray(y, float)
    mu = np.asarray(mu, float)
    sigma = np.asarray(sigma, float)
    out = np.full_like(y, np.nan, dtype=float)
    mask = np.isfinite(y) & np.isfinite(mu) & np.isfinite(sigma) & (sigma > 0)
    if not np.any(mask):
        return out
    yy = y[mask]
    mm = mu[mask]
    ss = np.clip(sigma[mask], 1e-8, None)
    z = (yy - mm) / ss
    crps = ss * (z * (2 * norm.cdf(z) - 1) + 2 * norm.pdf(z) - 1 / np.sqrt(np.pi))
    out[mask] = crps
    return out


def _compute_bins_and_means(y: np.ndarray, mu: np.ndarray, sigma_total: np.ndarray, *, n_bins: int) -> pd.DataFrame:
    y = np.asarray(y, float)
    mu = np.asarray(mu, float)
    sigma_total = np.asarray(sigma_total, float)
    abs_err = np.abs(y - mu)
    crps = _gaussian_crps_per_row(y, mu, sigma_total)
    mask = np.isfinite(sigma_total) & (sigma_total > 0) & np.isfinite(abs_err)
    if not np.any(mask):
        return pd.DataFrame(columns=["bin", "bin_sigma_mean", "bin_mae_mean", "bin_crps_mean", "n"])
    s = sigma_total[mask]
    ae = abs_err[mask]
    cr = crps[mask]
    bins = pd.qcut(s, n_bins, labels=False, duplicates="drop")
    df = pd.DataFrame({"bin": bins.astype(int) + 1, "sigma": s, "abs_err": ae, "crps": cr})
    out = (
        df.groupby("bin")
        .agg(
            bin_sigma_mean=("sigma", "mean"),
            bin_mae_mean=("abs_err", "mean"),
            bin_crps_mean=("crps", "mean"),
            n=("abs_err", "size"),
        )
        .reset_index()
        .sort_values("bin")
        .reset_index(drop=True)
    )
    return out


bins_rows = []
for spec in METHODS:
    if spec.head_type == "point" or spec.kind == "dido":
        continue
    try:
        df = method_id_frame(spec, "test")
    except Exception as e:
        print(f"[skip bins] {spec.key}: {type(e).__name__}: {e}")
        continue

    y = pd.to_numeric(df.get("y_true"), errors="coerce").to_numpy(float)
    mu = pd.to_numeric(df.get("mu"), errors="coerce").to_numpy(float)
    s_a_raw = pd.to_numeric(df.get("sigma_ale_raw"), errors="coerce").to_numpy(float)
    s_e_raw = pd.to_numeric(df.get("sigma_epi_raw"), errors="coerce").fillna(0.0).to_numpy(float)

    if USE_CALIBRATED_SIGMAS_LOCAL:
        _, _, s_t = calibrated_sigmas(
            head_type=spec.head_type,
            ale_src=spec.aleatoric_source,
            epi_src=spec.epistemic_source,
            sigma_ale_raw=s_a_raw,
            sigma_epi_raw=s_e_raw,
        )
    else:
        s_t = np.sqrt(np.maximum(np.nan_to_num(s_a_raw, nan=0.0) ** 2 + np.nan_to_num(s_e_raw, nan=0.0) ** 2, 0.0))

    bdf = _compute_bins_and_means(y, mu, s_t, n_bins=N_BINS)
    if bdf.empty:
        continue
    bdf["method"] = spec.label
    bins_rows.append(bdf)

if not bins_rows:
    print("[bins] No binned data available.")
else:
    df_bins = pd.concat(bins_rows, ignore_index=True)
    metric_key = str(BIN_METRIC).strip().lower()
    if metric_key not in {"mae", "crps"}:
        raise ValueError(f"BIN_METRIC must be 'mae' or 'crps', got {BIN_METRIC!r}")
    y_col = "bin_mae_mean" if metric_key == "mae" else "bin_crps_mean"
    y_label = "Mean absolute error (MAE)" if metric_key == "mae" else "Mean Gaussian CRPS"
    EXPORTER.export_table(df_bins, f"binned_error_{metric_key}", dataset=DATASET_KEY, split="test")

    methods_order = df_bins["method"].dropna().unique().tolist()
    if methods_order:
        y_vals = df_bins[y_col].to_numpy(float)
        y_vals = y_vals[np.isfinite(y_vals)]
        y_min, y_max = float(np.min(y_vals)), float(np.max(y_vals))
        pad = 0.05 * (y_max - y_min) if y_max > y_min else 1.0
        y_lim = (y_min - pad, y_max + pad)

        ncols = 3
        nrows = int(np.ceil(len(methods_order) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows), sharex=False, sharey=False)
        axes = np.atleast_1d(axes).reshape(-1)

        for ax, method in zip(axes, methods_order):
            sub = df_bins[df_bins["method"] == method].copy()
            if sub.empty:
                ax.set_axis_off()
                continue
            ax.plot(sub["bin"], sub[y_col], "-o", color="tab:blue")
            ax.set_ylim(*y_lim)
            ax.set_title(method)
            ax.set_xlabel("Uncertainty bin (1 = lowest sigma, 10 = highest sigma)")
            ax.set_ylabel(y_label)
            ax.set_xticks(np.arange(1, N_BINS + 1))
            ax.grid(True, alpha=0.2)

        for ax in axes[len(methods_order):]:
            ax.set_axis_off()

        fig.suptitle(f"Binned error vs predicted uncertainty (ID:test) - y={metric_key.upper()}")
        fig.tight_layout()
        savefig(fig, f"binned_error_{metric_key}", dataset=DATASET_KEY, split="test")
        plt.show()


## Phase 5: OOD + large-error detection

This phase evaluates **ID vs OOD separation** and **large-error detection** using uncertainty scores.

In [ ]:
# Phase 5A: OOD shift + AUROC + gains curves (file-first)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

USE_CALIBRATED_SIGMAS_LOCAL = USE_CALIBRATED_SIGMAS if "USE_CALIBRATED_SIGMAS" in globals() else True
OOD_LABEL = OOD_LABELS[0] if OOD_LABELS else None

DIDO_CANDIDATES = ["dido_strength_raw", "dido_entropy_raw", "dido_vacuity_raw"]
DIDO_SIGNAL = settings.get("dido_signal", DIDO_CANDIDATES[0])
DIDO_ALLOW_FLIP = True


def _to_float_arr(df: pd.DataFrame, col: str) -> np.ndarray:
    if col not in df.columns:
        return np.full(len(df), np.nan, dtype=float)
    return pd.to_numeric(df[col], errors="coerce").to_numpy(float)


def _sigmas_for(spec: MethodSpec, df: pd.DataFrame):
    s_a = _to_float_arr(df, "sigma_ale_raw")
    s_e = _to_float_arr(df, "sigma_epi_raw")
    if USE_CALIBRATED_SIGMAS_LOCAL:
        s_a, s_e, s_t = calibrated_sigmas(
            head_type=spec.head_type,
            ale_src=spec.aleatoric_source,
            epi_src=spec.epistemic_source,
            sigma_ale_raw=s_a,
            sigma_epi_raw=s_e,
        )
    else:
        s_t = np.sqrt(np.nan_to_num(s_a, nan=0.0) ** 2 + np.nan_to_num(s_e, nan=0.0) ** 2)
    return {"Ua": s_a, "Ue": s_e, "Utot": s_t}


def _pick_dido_col(df: pd.DataFrame) -> str | None:
    if DIDO_SIGNAL in df.columns:
        return DIDO_SIGNAL
    for c in DIDO_CANDIDATES:
        if c in df.columns:
            return c
    return None


def _auroc(id_scores: np.ndarray, ood_scores: np.ndarray):
    y = np.concatenate([np.zeros_like(id_scores), np.ones_like(ood_scores)])
    scores = np.concatenate([id_scores, ood_scores])
    mask = np.isfinite(scores) & np.isfinite(y)
    y = y[mask]
    scores = scores[mask]
    if len(y) == 0 or len(np.unique(y)) < 2:
        return float("nan")
    if np.nanstd(scores) == 0:
        return float("nan")
    return float(roc_auc_score(y, scores))


if not OOD_LABEL:
    print("[phase5] No OOD labels configured; skipping OOD analyses.")
else:
    shift_rows = []
    auc_rows = []
    gains = []

    # Only methods with epistemic signals (mc/ensemble) + DIDO
    epi_methods = [
        m for m in METHODS
        if (m.kind == "dido") or (str(m.epistemic_source).lower() in {"mc", "ensemble"})
    ]

    for spec in epi_methods:
        try:
            df_id = method_id_frame(spec, "test")
            df_ood = method_ood_frame(spec, OOD_LABEL)
        except Exception as e:
            print(f"[skip OOD] {spec.key}: {type(e).__name__}: {e}")
            continue

        if spec.kind == "dido":
            col = _pick_dido_col(df_id)
            if col is None or col not in df_ood.columns:
                print(f"[skip OOD] {spec.key}: no dido score column")
                continue
            s_id = _to_float_arr(df_id, col)
            s_ood = _to_float_arr(df_ood, col)
            auc = _auroc(s_id, s_ood)
            if DIDO_ALLOW_FLIP and np.isfinite(auc) and auc < 0.5:
                auc = 1.0 - auc
                s_id, s_ood = -s_id, -s_ood
            auc_rows.append({"method": spec.label, "key": spec.key, "signal": col, "auroc": auc})

            # Gains curve for DIDO
            scores = np.concatenate([s_id, s_ood])
            y = np.concatenate([np.zeros_like(s_id), np.ones_like(s_ood)])
            mask = np.isfinite(scores) & np.isfinite(y)
            scores = scores[mask]
            y = y[mask]
            if len(y) and y.sum() > 0:
                order = np.argsort(scores)[::-1]
                y_sorted = y[order]
                recall = np.cumsum(y_sorted) / y.sum()
                coverage = (np.arange(len(y_sorted)) + 1) / len(y_sorted)
                gains.append((spec.label + " (DIDO)", coverage, recall))
            continue

        sig_id = _sigmas_for(spec, df_id)
        sig_ood = _sigmas_for(spec, df_ood)

        for sig_name in ["Ue", "Utot"]:
            s_id = sig_id[sig_name]
            s_ood = sig_ood[sig_name]

            if not np.isfinite(s_id).any() and not np.isfinite(s_ood).any():
                continue

            shift_rows.append({
                "method": spec.label,
                "key": spec.key,
                "signal": sig_name,
                "id_median": float(np.nanmedian(s_id)),
                "ood_median": float(np.nanmedian(s_ood)),
                "delta": float(np.nanmedian(s_ood) - np.nanmedian(s_id)),
            })

            auc_rows.append({
                "method": spec.label,
                "key": spec.key,
                "signal": sig_name,
                "auroc": _auroc(s_id, s_ood),
            })

            if sig_name == "Utot":
                scores = np.concatenate([s_id, s_ood])
                y = np.concatenate([np.zeros_like(s_id), np.ones_like(s_ood)])
                mask = np.isfinite(scores) & np.isfinite(y)
                scores = scores[mask]
                y = y[mask]
                if len(y) and y.sum() > 0:
                    order = np.argsort(scores)[::-1]
                    y_sorted = y[order]
                    recall = np.cumsum(y_sorted) / y.sum()
                    coverage = (np.arange(len(y_sorted)) + 1) / len(y_sorted)
                    gains.append((spec.label, coverage, recall))

    if shift_rows:
        shift_df = pd.DataFrame(shift_rows)
        print(shift_df.pivot_table(index="method", columns="signal", values="delta", aggfunc="mean"))

    if auc_rows:
        auc_df = pd.DataFrame(auc_rows).sort_values(["signal", "auroc"], ascending=[True, False])
        EXPORTER.export_table(auc_df, f"ood_detection_auroc_{OOD_LABEL}", dataset=DATASET_KEY, split="test")
        display(auc_df)

    if gains:
        gains_map = {}
        for name, cov, rec in gains:
            gains_map.setdefault(name, []).append((cov, rec))

        methods = list(gains_map.keys())
        ncols = 2
        nrows = int(np.ceil(len(methods) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(7.5 * ncols, 5.0 * nrows), sharex=False, sharey=False)
        axes = np.atleast_1d(axes).reshape(-1)

        for ax, method in zip(axes, methods):
            ax.plot([0, 1], [0, 1], "--", alpha=0.35, color="0.5", label="random")
            for cov, rec in gains_map[method]:
                ax.plot(cov, rec, lw=2, label="Utot")
            ax.set_title(method)
            ax.set_xlabel("Coverage (sorted by score)")
            ax.set_ylabel("OOD recall")
            ax.set_ylim(0, 1.05)
            ax.grid(alpha=0.2)
            ax.legend(frameon=False, fontsize=8)

        for ax in axes[len(methods):]:
            ax.set_axis_off()

        fig.suptitle(f"OOD detection gains (label={OOD_LABEL})")
        fig.tight_layout(rect=[0, 0.03, 1, 0.98])
        savefig(fig, f"ood_detection_gains_{OOD_LABEL}", dataset=DATASET_KEY, split="test")
        plt.show()


In [ ]:
# Phase 5A1: kNN score density (ID vs OOD)

import pandas as pd
import matplotlib.pyplot as plt

if not DATASET_KEY:
    print("[knn density] DATASET_KEY missing; skipping.")
else:
    knn_dir = PROJECT_ROOT / "datasets" / "knn_ood"
    id_path = knn_dir / f"{DATASET_KEY}_id_knn.csv"
    ood_path = knn_dir / f"{DATASET_KEY}_ood_knn.csv"

    if not id_path.exists() or not ood_path.exists():
        print(f"[knn density] Missing files: {id_path} or {ood_path}")
    else:
        def _read_knn(path):
            cols = pd.read_csv(path, nrows=0).columns
            if "knn_score" not in cols:
                return None
            return pd.read_csv(path, usecols=["knn_score"])

        df_id = _read_knn(id_path)
        df_ood = _read_knn(ood_path)
        if df_id is None or df_ood is None:
            print("[knn density] 'knn_score' column missing; skipping.")
        else:
            plt.figure(figsize=(7, 4.5))
            plt.hist(df_id["knn_score"].to_numpy(float), bins=50, density=True, alpha=0.6, label="ID", color="tab:blue")
            plt.hist(df_ood["knn_score"].to_numpy(float), bins=50, density=True, alpha=0.6, label="OOD", color="tab:orange")
            plt.xlabel("knn_score")
            plt.ylabel("density")
            plt.title("kNN score density")
            plt.legend(frameon=False)
            plt.grid(True, alpha=0.2)
            plt.tight_layout()
            savefig(plt.gcf(), "knn_score_density", dataset=DATASET_KEY, split="test")
            plt.show()


In [ ]:
# Phase 5A2: OOD - ID quantile shift (Ua / Ue)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

INCLUDE_ALEATORIC_ONLY_BASELINES = False
USE_CALIBRATED_SIGMAS_LOCAL = USE_CALIBRATED_SIGMAS if "USE_CALIBRATED_SIGMAS" in globals() else True
QUANTILES = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

if not OOD_LABELS:
    print("[quantile shift] No OOD labels configured; skipping.")
else:
    def _has_epi(spec: MethodSpec) -> bool:
        return str(spec.epistemic_source).lower() in {"mc", "ensemble"}

    def _sigmas_for(spec: MethodSpec, df: pd.DataFrame):
        s_a_raw = pd.to_numeric(df.get("sigma_ale_raw"), errors="coerce").to_numpy(float)
        s_e_raw = pd.to_numeric(df.get("sigma_epi_raw"), errors="coerce").to_numpy(float)
        if USE_CALIBRATED_SIGMAS_LOCAL:
            s_e0 = np.nan_to_num(s_e_raw, nan=0.0)
            s_a, s_e, s_t = calibrated_sigmas(
                head_type=spec.head_type,
                ale_src=spec.aleatoric_source,
                epi_src=spec.epistemic_source,
                sigma_ale_raw=s_a_raw,
                sigma_epi_raw=s_e0,
            )
            s_e = np.asarray(s_e, float)
            s_e[~np.isfinite(s_e_raw)] = np.nan
            return np.asarray(s_a, float), s_e, np.asarray(s_t, float)
        s_a = np.asarray(s_a_raw, float)
        s_e = np.asarray(s_e_raw, float)
        s_t = np.sqrt(np.maximum(np.nan_to_num(s_a, nan=0.0) ** 2 + np.nan_to_num(s_e, nan=0.0) ** 2, 0.0))
        return s_a, s_e, s_t

    def _quantile_row(vals: np.ndarray) -> dict:
        vals = np.asarray(vals, float)
        vals = vals[np.isfinite(vals)]
        if vals.size == 0:
            return {f"q{int(q*100)}": np.nan for q in QUANTILES}
        return {f"q{int(q*100)}": float(np.quantile(vals, q)) for q in QUANTILES}

    methods_used = []
    for spec in METHODS:
        if spec.head_type == "point" or spec.kind == "dido":
            continue
        if _has_epi(spec) or INCLUDE_ALEATORIC_ONLY_BASELINES:
            methods_used.append(spec)

    if not methods_used:
        print("[quantile shift] No eligible methods.")
    else:
        for sig_key, sig_title in [("Ua", "Ua (aleatoric)"), ("Ue", "Ue (epistemic)")]:
            ncols = 2
            nrows = int(np.ceil(len(methods_used) / ncols))
            fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 3.8 * nrows), sharex=False, sharey=False)
            axes = np.atleast_1d(axes).reshape(-1)

            for ax, spec in zip(axes, methods_used):
                try:
                    df_id = method_id_frame(spec, "test")
                    sA_id, sE_id, _ = _sigmas_for(spec, df_id)
                except Exception as e:
                    ax.set_axis_off()
                    ax.set_title(f"{spec.label} (skip ID: {type(e).__name__})")
                    continue

                sA_ood_list = []
                sE_ood_list = []
                for label in OOD_LABELS:
                    try:
                        df_o = method_ood_frame(spec, label)
                        sA_o, sE_o, _ = _sigmas_for(spec, df_o)
                        sA_ood_list.append(sA_o)
                        sE_ood_list.append(sE_o)
                    except Exception:
                        continue

                if sA_ood_list:
                    sA_ood = np.concatenate(sA_ood_list)
                    sE_ood = np.concatenate(sE_ood_list)
                else:
                    sA_ood = np.array([])
                    sE_ood = np.array([])

                id_vals = sA_id if sig_key == "Ua" else sE_id
                ood_vals = sA_ood if sig_key == "Ua" else sE_ood

                q_id = _quantile_row(id_vals)
                q_ood = _quantile_row(ood_vals)
                delta = [q_ood[f"q{int(q*100)}"] - q_id[f"q{int(q*100)}"] for q in QUANTILES]

                x = np.arange(len(QUANTILES))
                ax.plot(x, delta, "o-", label="OOD - ID", color="tab:purple")
                ax.axhline(0.0, color="k", lw=1, alpha=0.4)
                ax.set_xticks(x)
                ax.set_xticklabels([f"q{int(q*100)}" for q in QUANTILES])
                ax.set_title(spec.label)
                ax.set_ylabel(f"Delta {sig_key}")
                ax.grid(True, alpha=0.2)
                ax.legend(frameon=False)

            for ax in axes[len(methods_used):]:
                ax.set_axis_off()

            fig.suptitle(f"{sig_title} quantile shift (OOD - ID) - {'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}")
            savefig(fig, f"ood_quantile_shift_{sig_key.lower()}_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split="test")
            plt.show()


In [ ]:
# Phase 5A3: OOD detection AUROC/AUPRC + gains curves

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score

USE_CALIBRATED_SIGMAS_LOCAL = USE_CALIBRATED_SIGMAS if "USE_CALIBRATED_SIGMAS" in globals() else True
OOD_LABELS_LOCAL = list(OOD_LABELS) if "OOD_LABELS" in globals() else []
SELECT_SIGNALS = ["Ua", "Ue", "Utot"]
SIG_COLORS_LOCAL = (
    SIG_COLORS
    if "SIG_COLORS" in globals()
    else {"Ua": "tab:blue", "Ue": "tab:green", "Utot": "tab:orange", "blend": "tab:purple", "random": "0.5"}
)

INCLUDE_DIDO = True
DIDO_SIGNAL = settings.get("dido_signal", "dido_strength_raw")
DIDO_CANDIDATES = ["dido_entropy_raw", "dido_vacuity_raw", "dido_strength_raw"]
DIDO_ALLOW_FLIP = True


def _label_candidates(label: str) -> list[str]:
    c = [label]
    if label.endswith("_hard"):
        c.append(label.replace("_hard", ""))
    return list(dict.fromkeys(c))


def _ood_metrics(y_true_bin: np.ndarray, score: np.ndarray):
    y_true_bin = np.asarray(y_true_bin, int)
    score = np.asarray(score, float)
    mask = np.isfinite(score) & np.isfinite(y_true_bin)
    if not np.any(mask):
        return float("nan"), float("nan"), 0
    y = y_true_bin[mask]
    s = score[mask]
    if len(np.unique(y)) < 2:
        return float("nan"), float("nan"), int(mask.sum())
    return float(roc_auc_score(y, s)), float(average_precision_score(y, s)), int(mask.sum())


def _gains_curve(y_ood: np.ndarray, score: np.ndarray):
    y_ood = np.asarray(y_ood, int)
    score = np.asarray(score, float)
    m = np.isfinite(score) & np.isfinite(y_ood)
    if not np.any(m):
        return None
    y = y_ood[m]
    s = score[m]
    if len(np.unique(y)) < 2:
        return None
    order = np.argsort(-s, kind="mergesort")
    y = y[order]
    total_pos = float(np.sum(y))
    if total_pos <= 0:
        return None
    cum_pos = np.cumsum(y).astype(float)
    recall = cum_pos / total_pos
    coverage = (np.arange(1, len(y) + 1) / float(len(y))).astype(float)
    aurc = float(np.trapezoid(recall, coverage))
    return coverage, recall, aurc


def _sigmas(spec: MethodSpec, df: pd.DataFrame):
    s_a_raw = pd.to_numeric(df.get("sigma_ale_raw"), errors="coerce").to_numpy(float)
    s_e_raw = pd.to_numeric(df.get("sigma_epi_raw"), errors="coerce").to_numpy(float)
    miss_epi = ~np.isfinite(s_e_raw)
    s_e0 = np.nan_to_num(s_e_raw, nan=0.0)

    if USE_CALIBRATED_SIGMAS_LOCAL:
        s_a, s_e, s_t = calibrated_sigmas(
            head_type=spec.head_type,
            ale_src=spec.aleatoric_source,
            epi_src=spec.epistemic_source,
            sigma_ale_raw=s_a_raw,
            sigma_epi_raw=s_e0,
        )
        s_e = np.asarray(s_e, float)
        s_e[miss_epi] = np.nan
        return np.asarray(s_a, float), s_e, np.asarray(s_t, float)

    s_a = np.asarray(s_a_raw, float)
    s_e = np.asarray(s_e_raw, float)
    s_t = np.sqrt(np.maximum(np.nan_to_num(s_a, nan=0.0) ** 2 + np.nan_to_num(s_e, nan=0.0) ** 2, 0.0))
    return s_a, s_e, s_t


def _load_ood_for_spec(spec: MethodSpec, ood_label: str) -> pd.DataFrame:
    last_err = None
    for lab in _label_candidates(ood_label):
        try:
            return method_ood_frame(spec, lab)
        except Exception as e:
            last_err = e
    raise last_err


base_specs = [m for m in METHODS if m.kind in {"regression", "ensemble"} and str(m.epistemic_source).lower() in {"mc", "ensemble"}]
rows = []
gains_rows = []

for ood_label in OOD_LABELS_LOCAL:
    for spec in base_specs:
        try:
            df_id = method_id_frame(spec, "test")
            df_ood = _load_ood_for_spec(spec, ood_label)
        except Exception as e:
            print(f"[skip] {spec.label} / ood={ood_label}: {type(e).__name__}: {e}")
            continue

        df_id = df_id.copy()
        df_ood = df_ood.copy()
        df_id["is_ood"] = 0
        df_ood["is_ood"] = 1
        df_all = pd.concat([df_id, df_ood], ignore_index=True)
        ybin = df_all["is_ood"].to_numpy(int)

        s_a, s_e, s_t = _sigmas(spec, df_all)
        sig_map = {"Ua": s_a, "Ue": s_e, "Utot": s_t}

        for sig_name in SELECT_SIGNALS:
            score = sig_map[sig_name]
            auroc, auprc, n_used = _ood_metrics(ybin, score)
            rows.append({
                "ood_label": ood_label,
                "method": spec.label,
                "signal": sig_name,
                "n_id": int(len(df_id)),
                "n_ood": int(len(df_ood)),
                "n_used": n_used,
                "auroc": auroc,
                "auprc": auprc,
                "use_calibrated": bool(USE_CALIBRATED_SIGMAS_LOCAL),
            })

            g = _gains_curve(ybin, score)
            if g is not None:
                cov, rec, aurc = g
                gains_rows.append({
                    "ood_label": ood_label,
                    "method": spec.label,
                    "signal": sig_name,
                    "coverage": cov,
                    "recall": rec,
                    "aurc": aurc,
                })

    dido_spec = next((m for m in METHODS if m.kind == "dido" and m.key == "dido_lpl"), None)
    if INCLUDE_DIDO and dido_spec is not None:
        try:
            df_id_d = method_id_frame(dido_spec, "test")
            df_ood_d = _load_ood_for_spec(dido_spec, ood_label)
        except Exception as e:
            print(f"[skip] Laplace DIDO / ood={ood_label}: {type(e).__name__}: {e}")
        else:
            df_id_d = df_id_d.copy()
            df_ood_d = df_ood_d.copy()
            df_id_d["is_ood"] = 0
            df_ood_d["is_ood"] = 1
            df_all_d = pd.concat([df_id_d, df_ood_d], ignore_index=True)
            ybin = df_all_d["is_ood"].to_numpy(int)

            if str(DIDO_SIGNAL).lower() == "auto":
                dido_cols = [c for c in DIDO_CANDIDATES if c in df_all_d.columns]
            else:
                dido_cols = [str(DIDO_SIGNAL)] if str(DIDO_SIGNAL) in df_all_d.columns else []

            if not dido_cols:
                print(f"[skip] Laplace DIDO / ood={ood_label}: dido column not found")
            else:
                best = None
                for col in dido_cols:
                    score0 = pd.to_numeric(df_all_d[col], errors="coerce").to_numpy(float)
                    auroc, auprc, n_used = _ood_metrics(ybin, score0)
                    flipped = False
                    score_used = score0
                    if DIDO_ALLOW_FLIP:
                        auroc_neg, auprc_neg, _ = _ood_metrics(ybin, -score0)
                        if np.isfinite(auroc_neg) and (not np.isfinite(auroc) or auroc_neg > auroc):
                            auroc, auprc, flipped = auroc_neg, auprc_neg, True
                            score_used = -score0

                    cand = (float(auroc), float(auprc), col, flipped, int(n_used), score_used)
                    if best is None or (np.isfinite(cand[0]) and (not np.isfinite(best[0]) or cand[0] > best[0])):
                        best = cand

                if best is not None:
                    auroc, auprc, col, flipped, n_used, score_used = best
                    sig_name = col + (" (flipped)" if flipped else "")
                    rows.append({
                        "ood_label": ood_label,
                        "method": "Laplace + DIDO",
                        "signal": sig_name,
                        "n_id": int(len(df_id_d)),
                        "n_ood": int(len(df_ood_d)),
                        "n_used": int(n_used),
                        "auroc": auroc,
                        "auprc": auprc,
                        "use_calibrated": False,
                    })
                    g = _gains_curve(ybin, score_used)
                    if g is not None:
                        cov, rec, aurc = g
                        gains_rows.append({
                            "ood_label": ood_label,
                            "method": "Laplace + DIDO",
                            "signal": sig_name,
                            "coverage": cov,
                            "recall": rec,
                            "aurc": aurc,
                        })

if rows:
    df_ooddet = pd.DataFrame(rows)
    EXPORTER.export_table(df_ooddet, f"ood_detection_full_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split="test")
    display(df_ooddet.sort_values(["ood_label", "auroc"], ascending=[True, False]).reset_index(drop=True))

    for ood_label in OOD_LABELS_LOCAL:
        sub = df_ooddet[df_ooddet["ood_label"] == ood_label].copy()
        sub = sub[np.isfinite(sub["auroc"].to_numpy(float))].copy()
        if sub.empty:
            continue
        sub["name"] = sub["method"] + " / " + sub["signal"]
        sub = sub.sort_values("auroc", ascending=True)

        fig, ax = plt.subplots(figsize=(9, max(3.5, 0.35 * len(sub))))
        y = np.arange(len(sub))
        vals = sub["auroc"].to_numpy(float)
        ax.barh(y, vals, alpha=0.85)
        ax.set_yticks(y)
        ax.set_yticklabels(sub["name"].tolist())
        ax.set_xlabel("AUROC (ID vs OOD)")
        ax.set_title(
            f"OOD detection AUROC - {ood_label} ({'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'})"
            + f"  signals={','.join(SELECT_SIGNALS)}"
        )
        ax.grid(True, axis="x", alpha=0.2)
        ax.ticklabel_format(style="plain", axis="x", useOffset=False)
        xmax = float(vals.max()) if len(vals) else 0.0
        pad = 0.02 * xmax
        for yi, v in zip(y, vals):
            ax.text(max(v - pad, 0.0), yi, f"{v:.3f}", va="center", ha="right", fontsize=9)
        fig.tight_layout()
        fig.subplots_adjust(left=0.45)
        savefig(fig, f"ood_detection_auroc_{ood_label}_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split="test")
        plt.show()

    gains_map = {(r["ood_label"], r["method"], r["signal"]): (r["coverage"], r["recall"], r["aurc"]) for r in gains_rows}

    for ood_label in OOD_LABELS_LOCAL:
        methods = sorted({m for (ol, m, _s) in gains_map.keys() if ol == ood_label})
        if not methods:
            continue

        ncols = 2
        nrows = int(np.ceil(len(methods) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(7.5 * ncols, 5.0 * nrows), sharex=False, sharey=False, constrained_layout=True)
        axes = np.atleast_1d(axes).reshape(-1)

        for ax, method in zip(axes, methods):
            ax.plot([0, 1], [0, 1], "--", alpha=0.35, color=SIG_COLORS_LOCAL.get("random", "0.5"), label="random")

            sigs_here = [sig for (ol, m, sig) in gains_map.keys() if (ol == ood_label and m == method)]
            for sig in sigs_here:
                cov, rec, aurc = gains_map[(ood_label, method, sig)]
                color = SIG_COLORS_LOCAL.get(sig, None) if sig in {"Ua", "Ue", "Utot"} else None
                ax.plot(cov, rec, lw=2, color=color, label=f"{sig} (AURC={aurc:.3f})")

            ax.set_title(method)
            ax.set_xlabel("Coverage reviewed (fraction, highest score first)")
            ax.set_ylabel("Recall of OOD")
            ax.set_ylim(0, 1.05)
            ax.grid(True, alpha=0.2)
            ax.legend(frameon=False)

        for ax in axes[len(methods):]:
            ax.set_axis_off()

        fig.suptitle(
            f"OOD gains curves - {ood_label} ({'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'})",
            y=1.02,
        )
        savefig(fig, f"ood_gains_curves_{ood_label}_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split="test")
        plt.show()


In [ ]:
# Phase 5B: large-error detection (ID:test) using uncertainty scores

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

USE_CALIBRATED_SIGMAS_LOCAL = USE_CALIBRATED_SIGMAS if "USE_CALIBRATED_SIGMAS" in globals() else True

rows = []
gains = []

for spec in METHODS:
    if spec.kind == "dido":
        continue
    try:
        df = method_id_frame(spec, "test")
    except Exception as e:
        print(f"[skip large-error] {spec.key}: {type(e).__name__}: {e}")
        continue
    if "y_true" not in df.columns or "mu" not in df.columns:
        print(f"[skip large-error] {spec.key}: missing y_true/mu")
        continue

    y = pd.to_numeric(df["y_true"], errors="coerce").to_numpy(float)
    mu = pd.to_numeric(df["mu"], errors="coerce").to_numpy(float)
    err = np.abs(y - mu)
    is_large = err > float(TAU)

    s_a, s_e, s_t = _sigmas_for(spec, df)
    sig_map = {"Ua": s_a, "Ue": s_e, "Utot": s_t}
    for sig_name in ["Ua", "Ue", "Utot"]:
        s = sig_map[sig_name]
        mask = np.isfinite(s) & np.isfinite(err)
        if mask.sum() == 0 or len(np.unique(is_large[mask])) < 2:
            continue
        auroc = float(roc_auc_score(is_large[mask], s[mask]))
        rows.append({"method": spec.label, "key": spec.key, "signal": sig_name, "auroc": auroc})

        if sig_name == "Utot":
            order = np.argsort(s[mask])[::-1]
            y_sorted = is_large[mask][order].astype(float)
            if y_sorted.sum() > 0:
                recall = np.cumsum(y_sorted) / y_sorted.sum()
                coverage = (np.arange(len(y_sorted)) + 1) / len(y_sorted)
                gains.append((spec.label, coverage, recall))

if rows:
    df_auroc = pd.DataFrame(rows).sort_values(["signal", "auroc"], ascending=[True, False])
    EXPORTER.export_table(df_auroc, "large_error_detection_auroc", dataset=DATASET_KEY, split="test")
    display(df_auroc)

if gains:
    plt.figure(figsize=(7, 5))
    for name, cov, rec in gains:
        plt.plot(cov, rec, label=name)
    plt.xlabel("Coverage (sorted by score)")
    plt.ylabel("Large-error recall")
    plt.title(f"Large-error detection gains (tau={TAU:.3g})")
    plt.legend(fontsize=8)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    savefig(plt.gcf(), "large_error_detection_gains", dataset=DATASET_KEY, split="test")
    plt.show()


In [ ]:
# Phase 5C: large-error AUROC (ID:test)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

USE_CALIBRATED_SIGMAS_LOCAL = USE_CALIBRATED_SIGMAS if "USE_CALIBRATED_SIGMAS" in globals() else True

rows = []
specs = [m for m in METHODS if (m.kind == "nf" or str(m.epistemic_source).lower() in {"mc", "ensemble"})]

for spec in specs:
    if spec.kind == "dido" or spec.head_type == "point":
        continue
    try:
        df = method_id_frame(spec, "test")
    except Exception as e:
        print(f"[skip large AUROC] {spec.key}: {type(e).__name__}: {e}")
        continue
    if "y_true" not in df.columns or "mu" not in df.columns:
        continue

    y = pd.to_numeric(df["y_true"], errors="coerce").to_numpy(float)
    mu = pd.to_numeric(df["mu"], errors="coerce").to_numpy(float)
    err = np.abs(y - mu)
    is_large = err > float(TAU)

    s_a_raw = pd.to_numeric(df.get("sigma_ale_raw"), errors="coerce").to_numpy(float)
    s_e_raw = pd.to_numeric(df.get("sigma_epi_raw"), errors="coerce").to_numpy(float)
    miss_epi = ~np.isfinite(s_e_raw)
    s_e0 = np.nan_to_num(s_e_raw, nan=0.0)

    if USE_CALIBRATED_SIGMAS_LOCAL:
        s_a, s_e, s_t = calibrated_sigmas(
            head_type=spec.head_type,
            ale_src=spec.aleatoric_source,
            epi_src=spec.epistemic_source,
            sigma_ale_raw=s_a_raw,
            sigma_epi_raw=s_e0,
        )
        s_e = np.asarray(s_e, float)
        s_e[miss_epi] = np.nan
    else:
        s_a = np.asarray(s_a_raw, float)
        s_e = np.asarray(s_e_raw, float)
        s_t = np.sqrt(np.maximum(np.nan_to_num(s_a, nan=0.0) ** 2 + np.nan_to_num(s_e, nan=0.0) ** 2, 0.0))

    sigs = {"Ua": np.asarray(s_a, float), "Ue": np.asarray(s_e, float), "Utot": np.asarray(s_t, float)}

    for sig_name, s in sigs.items():
        if sig_name == "Ue" and str(spec.epistemic_source).lower() in {"none", "dido"}:
            continue
        mask = np.isfinite(s) & np.isfinite(err)
        if mask.sum() == 0 or len(np.unique(is_large[mask])) < 2:
            continue
        auroc = float(roc_auc_score(is_large[mask], s[mask]))
        rows.append({"method": spec.label, "key": spec.key, "signal": sig_name, "auroc": auroc})

if rows:
    df_le = pd.DataFrame(rows).sort_values("auroc", ascending=True)
    EXPORTER.export_table(df_le, f"large_error_auroc_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split="test")
    display(df_le)

    df_le["name"] = df_le["method"] + " / " + df_le["signal"]
    fig, ax = plt.subplots(figsize=(9, max(3.5, 0.35 * len(df_le))))
    y = np.arange(len(df_le))
    vals = df_le["auroc"].to_numpy(float)
    ax.barh(y, vals, alpha=0.85)
    ax.set_yticks(y)
    ax.set_yticklabels(df_le["name"].tolist())
    ax.set_xlabel("AUROC (large error vs not)")
    ax.set_title(f"Large-error detection AUROC - tau={TAU:.3g} (cal={USE_CALIBRATED_SIGMAS_LOCAL})")
    ax.grid(True, axis="x", alpha=0.2)
    ax.ticklabel_format(style="plain", axis="x", useOffset=False)
    xmax = float(vals.max()) if len(vals) else 0.0
    pad = 0.02 * xmax
    for yi, v in zip(y, vals):
        ax.text(max(v - pad, 0.0), yi, f"{v:.3f}", va="center", ha="right", fontsize=9)
    fig.tight_layout()
    fig.subplots_adjust(left=0.45)
    savefig(fig, f"large_error_auroc_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split="test")
    plt.show()
else:
    print("[large AUROC] No rows to plot.")


In [ ]:
# Phase 5D: large-error gains curves (Ua/Ue/Utot)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

USE_CALIBRATED_SIGMAS_LOCAL = USE_CALIBRATED_SIGMAS if "USE_CALIBRATED_SIGMAS" in globals() else True
EVAL_SCOPE_LOCAL = "id_test"  # "id_test" | "id_test+ood"
SIG_COLORS_LOCAL = (
    SIG_COLORS
    if "SIG_COLORS" in globals()
    else {"Ua": "tab:blue", "Ue": "tab:green", "Utot": "tab:orange", "blend": "tab:purple", "random": "0.5"}
)


def _sigmas(spec: MethodSpec, df: pd.DataFrame):
    s_a_raw = pd.to_numeric(df.get("sigma_ale_raw"), errors="coerce").to_numpy(float)
    s_e_raw = pd.to_numeric(df.get("sigma_epi_raw"), errors="coerce").to_numpy(float)
    miss_epi = ~np.isfinite(s_e_raw)
    s_e0 = np.nan_to_num(s_e_raw, nan=0.0)
    if USE_CALIBRATED_SIGMAS_LOCAL:
        s_a, s_e, s_t = calibrated_sigmas(
            head_type=spec.head_type,
            ale_src=spec.aleatoric_source,
            epi_src=spec.epistemic_source,
            sigma_ale_raw=s_a_raw,
            sigma_epi_raw=s_e0,
        )
        s_e = np.asarray(s_e, float)
        s_e[miss_epi] = np.nan
    else:
        s_a = np.asarray(s_a_raw, float)
        s_e = np.asarray(s_e_raw, float)
        s_t = np.sqrt(np.maximum(np.nan_to_num(s_a, nan=0.0) ** 2 + np.nan_to_num(s_e, nan=0.0) ** 2, 0.0))
    return np.asarray(s_a, float), np.asarray(s_e, float), np.asarray(s_t, float)


def _gains(score: np.ndarray, y_large: np.ndarray):
    d = pd.DataFrame({"score": score, "y": y_large}).dropna()
    if d.empty or d["y"].nunique() < 2:
        return None
    d = d.sort_values("score", ascending=False, kind="mergesort")
    total_pos = float(d["y"].sum())
    if total_pos == 0:
        return None
    cum_pos = d["y"].cumsum().to_numpy(float)
    recall = cum_pos / total_pos
    coverage = np.arange(1, len(d) + 1) / float(len(d))
    aurc = float(np.trapezoid(recall, coverage))
    return coverage, recall, aurc


def _auroc(y: np.ndarray, score: np.ndarray) -> float:
    y = np.asarray(y, int)
    score = np.asarray(score, float)
    m = np.isfinite(score) & np.isfinite(y)
    if not np.any(m):
        return float("nan")
    yy = y[m]
    ss = score[m]
    if np.unique(yy).size < 2:
        return float("nan")
    try:
        return float(roc_auc_score(yy, ss))
    except Exception:
        return float("nan")


def _load_for_scope(spec: MethodSpec) -> pd.DataFrame:
    parts = []
    df_id = method_id_frame(spec, "test").copy()
    df_id["dataset"] = "id:test"
    parts.append(df_id)

    if EVAL_SCOPE_LOCAL.strip().lower() == "id_test+ood":
        for ood_label in OOD_LABELS:
            try:
                df_o = method_ood_frame(spec, ood_label).copy()
                df_o["dataset"] = f"ood:{ood_label}"
                parts.append(df_o)
            except Exception as e:
                print(f"[skip] {spec.label} / ood={ood_label}: {type(e).__name__}: {e}")

    return pd.concat(parts, ignore_index=True)


specs = [m for m in METHODS if (m.kind == "nf" or str(m.epistemic_source).lower() in {"mc", "ensemble"})]
plots = []

for spec in specs:
    try:
        df_all = _load_for_scope(spec)
    except Exception as e:
        print(f"[skip large gains] {spec.label}: {type(e).__name__}: {e}")
        continue

    if "y_true" not in df_all.columns or "mu" not in df_all.columns:
        continue

    y = pd.to_numeric(df_all["y_true"], errors="coerce").to_numpy(float)
    mu = pd.to_numeric(df_all["mu"], errors="coerce").to_numpy(float)
    abs_err = np.abs(y - mu)
    y_large = (abs_err > float(TAU)).astype(int)

    s_a, s_e, s_t = _sigmas(spec, df_all)
    is_nf = spec.kind == "nf" or str(spec.epistemic_source).lower() == "none"

    if is_nf:
        valid = np.isfinite(s_a) & np.isfinite(s_t) & np.isfinite(y_large)
    else:
        valid = np.isfinite(s_a) & np.isfinite(s_e) & np.isfinite(s_t) & np.isfinite(y_large)

    if not np.any(valid):
        continue

    s_a, s_e, s_t, yL = s_a[valid], s_e[valid], s_t[valid], y_large[valid]

    g_ua = _gains(s_a, yL)
    g_ut = _gains(s_t, yL)
    g_ue = None if is_nf else _gains(s_e, yL)

    if g_ua is None or g_ut is None or (not is_nf and g_ue is None):
        continue

    auroc_ua = _auroc(yL, s_a)
    auroc_ut = _auroc(yL, s_t)
    auroc_ue = float("nan") if is_nf else _auroc(yL, s_e)

    curves = {
        "Ua": (g_ua[0], g_ua[1], g_ua[2], auroc_ua),
        "Utot": (g_ut[0], g_ut[1], g_ut[2], auroc_ut),
    }
    if g_ue is not None:
        curves["Ue"] = (g_ue[0], g_ue[1], g_ue[2], auroc_ue)

    plots.append({"title": spec.label, "kind": spec.kind, "curves": curves})

if not plots:
    print("[large gains] No curves to plot.")
else:
    n = len(plots)
    ncols = 2
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(7.5 * ncols, 5.0 * nrows), sharex=True, sharey=True, constrained_layout=True)
    axes = np.atleast_1d(axes).reshape(-1)

    for ax, item in zip(axes, plots):
        ax.plot([0, 1], [0, 1], linestyle="--", alpha=0.35,
            color=SIG_COLORS_LOCAL.get("random", "0.5"), label="random")

        for name, (cov, rec, aurc, auroc) in item["curves"].items():
            if item.get("kind") == "nf" and name != "Ua":
                continue
            ax.plot(cov, rec, lw=2, color=SIG_COLORS_LOCAL.get(name, None), label=f"{name} (AUROC={auroc:.3f}, AURC={aurc:.3f})")
        ax.set_title(item["title"])
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.05)
        ax.grid(True, alpha=0.2)
        ax.legend(frameon=False)

    for ax in axes[len(plots):]:
        ax.set_axis_off()

    fig.supxlabel("Coverage reviewed (fraction, highest score first)")
    fig.supylabel("Recall of large errors")
    fig.suptitle(f"Large-error gains (tau={TAU:.3g}) - scope={EVAL_SCOPE_LOCAL}")
    savefig(fig, f"large_error_gains_{EVAL_SCOPE_LOCAL}_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split="test")
    plt.show()


## Phase 6: OOD predictive quality + ID vs OOD metrics

Compute MAE/CRPS on OOD and compare ID:test vs OOD side-by-side.

In [ ]:
# Phase 6A: OOD predictive quality (MAE + CRPS) per method/label

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

USE_CALIBRATED_SIGMAS_LOCAL = USE_CALIBRATED_SIGMAS if "USE_CALIBRATED_SIGMAS" in globals() else True

if "gaussian_crps_mean" not in globals():
    from scipy.stats import norm as _norm
    def gaussian_crps_mean(y: np.ndarray, mu: np.ndarray, sigma: np.ndarray) -> float:
        y = np.asarray(y, float)
        mu = np.asarray(mu, float)
        sigma = np.asarray(sigma, float)
        mask = np.isfinite(y) & np.isfinite(mu) & np.isfinite(sigma) & (sigma > 0)
        if not np.any(mask):
            return float("nan")
        y = y[mask]
        mu = mu[mask]
        sigma = sigma[mask]
        z = (y - mu) / sigma
        crps = sigma * (z * (2 * _norm.cdf(z) - 1) + 2 * _norm.pdf(z) - 1 / np.sqrt(np.pi))
        return float(np.mean(crps))


def _sigma_total(spec: MethodSpec, df: pd.DataFrame) -> np.ndarray:
    s_a = pd.to_numeric(df.get("sigma_ale_raw"), errors="coerce").to_numpy(float)
    s_e = pd.to_numeric(df.get("sigma_epi_raw"), errors="coerce").fillna(0.0).to_numpy(float)
    if USE_CALIBRATED_SIGMAS_LOCAL:
        _, _, s_t = calibrated_sigmas(
            head_type=spec.head_type,
            ale_src=spec.aleatoric_source,
            epi_src=spec.epistemic_source,
            sigma_ale_raw=s_a,
            sigma_epi_raw=s_e,
        )
    else:
        s_t = np.sqrt(np.maximum(np.nan_to_num(s_a, nan=0.0) ** 2 + np.nan_to_num(s_e, nan=0.0) ** 2, 0.0))
    return np.asarray(s_t, float)


if not OOD_LABELS:
    print("[phase6] No OOD labels configured; skipping OOD quality.")
else:
    rows = []
    for label in OOD_LABELS:
        for spec in METHODS:
            if spec.kind == "dido":
                continue
            try:
                df = method_ood_frame(spec, label)
            except Exception as e:
                print(f"[skip OOD quality] {spec.key}/{label}: {type(e).__name__}: {e}")
                continue
            if "y_true" not in df.columns or "mu" not in df.columns:
                continue

            y = pd.to_numeric(df["y_true"], errors="coerce").to_numpy(float)
            mu = pd.to_numeric(df["mu"], errors="coerce").to_numpy(float)
            mae = float(np.nanmean(np.abs(y - mu)))
            rmse = float(np.sqrt(np.nanmean((y - mu) ** 2)))

            crps = float("nan")
            if spec.head_type != "point":
                s_t = _sigma_total(spec, df)
                crps = gaussian_crps_mean(y, mu, s_t)

            rows.append({
                "ood_label": label,
                "method": spec.label,
                "key": spec.key,
                "head_type": spec.head_type,
                "n": int(len(df)),
                "MAE": mae,
                "RMSE": rmse,
                "CRPS": crps,
            })

    df_ood_quality = pd.DataFrame(rows)
    EXPORTER.export_table(df_ood_quality, f"ood_quality_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split="test")
    display(df_ood_quality.sort_values(["ood_label", "MAE"]))

    def _barh(ax, df, metric, title):
        sub = df[np.isfinite(df[metric].to_numpy(float))].copy()
        if sub.empty:
            ax.set_axis_off()
            ax.set_title(f"{title} (no data)")
            return
        sub = sub.sort_values(metric, ascending=True)
        y = np.arange(len(sub))
        vals = sub[metric].to_numpy(float)
        ax.barh(y, vals, color="tab:blue", alpha=0.85)
        ax.set_yticks(y)
        ax.set_yticklabels(sub["method"])
        ax.invert_yaxis()
        ax.set_title(title)
        ax.grid(True, axis="x", alpha=0.2)
        ax.ticklabel_format(style="plain", axis="x", useOffset=False)

    for label in OOD_LABELS:
        sub = df_ood_quality[df_ood_quality["ood_label"] == label].copy()
        if sub.empty:
            continue
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        _barh(axes[0], sub, "MAE", f"MAE (OOD: {label})")
        sub_non_point = sub[sub["head_type"] != "point"]
        _barh(axes[1], sub_non_point, "CRPS", f"Gaussian CRPS (OOD: {label}) - non-point")
        plt.tight_layout()
        savefig(fig, f"ood_quality_{label}_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split="test")
        plt.show()


In [ ]:
# Phase 6B: ID:test vs OOD (MAE + CRPS) side-by-side

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

USE_CALIBRATED_SIGMAS_LOCAL = USE_CALIBRATED_SIGMAS if "USE_CALIBRATED_SIGMAS" in globals() else True

if not OOD_LABELS:
    print("[phase6] No OOD labels configured; skipping ID vs OOD plots.")
else:
    rows = []
    for label in OOD_LABELS:
        for spec in METHODS:
            if spec.kind == "dido":
                continue
            try:
                df_id = method_id_frame(spec, "test")
                df_ood = method_ood_frame(spec, label)
            except Exception as e:
                print(f"[skip ID/OOD] {spec.key}/{label}: {type(e).__name__}: {e}")
                continue
            if "y_true" not in df_id.columns or "mu" not in df_id.columns:
                continue

            def _metrics(df: pd.DataFrame):
                y = pd.to_numeric(df["y_true"], errors="coerce").to_numpy(float)
                mu = pd.to_numeric(df["mu"], errors="coerce").to_numpy(float)
                mae = float(np.nanmean(np.abs(y - mu)))
                crps = float("nan")
                if spec.head_type != "point":
                    s_t = _sigma_total(spec, df)
                    crps = gaussian_crps_mean(y, mu, s_t)
                return mae, crps

            mae_id, crps_id = _metrics(df_id)
            mae_ood, crps_ood = _metrics(df_ood)

            rows.append({
                "ood_label": label,
                "method": spec.label,
                "metric": "MAE",
                "id_value": mae_id,
                "ood_value": mae_ood,
            })
            rows.append({
                "ood_label": label,
                "method": spec.label,
                "metric": "CRPS",
                "id_value": crps_id,
                "ood_value": crps_ood,
            })

    df_id_ood = pd.DataFrame(rows)
    EXPORTER.export_table(df_id_ood, "id_vs_ood_metrics", dataset=DATASET_KEY, split="test")
    display(df_id_ood)

    for label in OOD_LABELS:
        for metric in ("MAE", "CRPS"):
            sub = df_id_ood[(df_id_ood["ood_label"] == label) & (df_id_ood["metric"] == metric)].copy()
            if sub.empty:
                continue
            if metric == "CRPS":
                sub = sub[np.isfinite(sub["id_value"]) & np.isfinite(sub["ood_value"])]
            x = np.arange(len(sub))
            width = 0.38
            fig, ax = plt.subplots(figsize=(10, 5))
            ax.bar(x - width / 2, sub["id_value"], width, label="ID:test")
            ax.bar(x + width / 2, sub["ood_value"], width, label=f"OOD:{label}")
            ax.set_xticks(x)
            ax.set_xticklabels(sub["method"], rotation=45, ha="right")
            ax.set_ylabel(metric)
            ax.set_title(f"ID vs OOD ({metric}) - label={label}")
            ax.legend()
            ax.grid(True, axis="y", alpha=0.2)
            ax.ticklabel_format(style="plain", axis="y", useOffset=False)
            fig.tight_layout()
            savefig(fig, f"id_vs_ood_{metric.lower()}_{label}", dataset=DATASET_KEY, split="test")
            plt.show()


In [ ]:
# Phase 6C: ID vs OOD metrics (barh, PDF style)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

USE_CALIBRATED_SIGMAS_LOCAL = USE_CALIBRATED_SIGMAS if "USE_CALIBRATED_SIGMAS" in globals() else True
OOD_LABEL_TO_PLOT = (OOD_LABELS[0] if ("OOD_LABELS" in globals() and len(OOD_LABELS) > 0) else "knn")


def _sigmas_total(spec: MethodSpec, df: pd.DataFrame) -> np.ndarray:
    s_a_raw = pd.to_numeric(df.get("sigma_ale_raw"), errors="coerce").to_numpy(float)
    s_e_raw = pd.to_numeric(df.get("sigma_epi_raw"), errors="coerce").fillna(0.0).to_numpy(float)
    if USE_CALIBRATED_SIGMAS_LOCAL:
        _, _, s_t = calibrated_sigmas(
            head_type=spec.head_type,
            ale_src=spec.aleatoric_source,
            epi_src=spec.epistemic_source,
            sigma_ale_raw=s_a_raw,
            sigma_epi_raw=s_e_raw,
        )
        return np.asarray(s_t, float)
    return np.sqrt(np.maximum(np.nan_to_num(s_a_raw, nan=0.0) ** 2 + np.nan_to_num(s_e_raw, nan=0.0) ** 2, 0.0))


def _compute_metrics(spec: MethodSpec, df: pd.DataFrame) -> dict:
    y = pd.to_numeric(df.get("y_true"), errors="coerce").to_numpy(float)
    mu = pd.to_numeric(df.get("mu"), errors="coerce").to_numpy(float)
    mae = float(np.nanmean(np.abs(y - mu))) if len(df) else float("nan")
    if spec.head_type == "point":
        return {"MAE": mae, "CRPS": float("nan")}
    s_t = _sigmas_total(spec, df)
    crps = gaussian_crps_mean(y, mu, s_t)
    return {"MAE": mae, "CRPS": crps}


SPECS = [m for m in METHODS if m.kind != "dido"]
rows = []
for spec in SPECS:
    try:
        df_id = method_id_frame(spec, "test")
        id_m = _compute_metrics(spec, df_id)
    except Exception as e:
        print(f"[skip] {spec.label} (ID): {type(e).__name__}: {e}")
        continue

    ood_m = {"MAE": float("nan"), "CRPS": float("nan")}
    try:
        df_ood = method_ood_frame(spec, OOD_LABEL_TO_PLOT)
        ood_m = _compute_metrics(spec, df_ood)
    except Exception as e:
        print(f"[skip] {spec.label} (OOD={OOD_LABEL_TO_PLOT}): {type(e).__name__}: {e}")

    rows.append({
        "method": spec.label,
        "head_type": spec.head_type,
        "mae_id": id_m["MAE"],
        "crps_id": id_m["CRPS"],
        "mae_ood": ood_m["MAE"],
        "crps_ood": ood_m["CRPS"],
    })

if rows:
    df_idood = pd.DataFrame(rows)
    EXPORTER.export_table(df_idood, f"id_vs_ood_summary_{OOD_LABEL_TO_PLOT}_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split="test")
    display(df_idood.sort_values("mae_id", ascending=True).reset_index(drop=True))

    def _grouped_barh(ax, labels, v_id, v_ood, title, fmt):
        y = np.arange(len(labels))
        h = 0.38
        ax.barh(y - h / 2, v_id, height=h, color="tab:blue", alpha=0.85, label="ID")
        ax.barh(y + h / 2, v_ood, height=h, color="tab:orange", alpha=0.85, label=f"OOD ({OOD_LABEL_TO_PLOT})")
        ax.set_yticks(y)
        ax.set_yticklabels(labels)
        ax.invert_yaxis()
        ax.set_title(title)
        ax.grid(True, axis="x", alpha=0.2)
        ax.ticklabel_format(style="plain", axis="x", useOffset=False)

        xmax = float(np.nanmax(np.concatenate([v_id, v_ood]))) if len(v_id) else 0.0
        pad = 0.02 * xmax
        for yi, (a, b) in enumerate(zip(v_id, v_ood)):
            if np.isfinite(a):
                ax.text(max(a - pad, 0.0), yi - h / 2, fmt(a), va="center", ha="right", fontsize=9)
            if np.isfinite(b):
                ax.text(max(b - pad, 0.0), yi + h / 2, fmt(b), va="center", ha="right", fontsize=9)

    order = df_idood.sort_values("mae_id", ascending=True)["method"].tolist()
    dfp = df_idood.set_index("method").reindex(order).reset_index()

    mae_id = dfp["mae_id"].to_numpy(float)
    mae_ood = dfp["mae_ood"].to_numpy(float)
    labels_mae = dfp["method"].tolist()

    mask_crps = np.isfinite(dfp["crps_id"].to_numpy(float)) | np.isfinite(dfp["crps_ood"].to_numpy(float))
    dfc = dfp[mask_crps].copy()
    crps_id = dfc["crps_id"].to_numpy(float)
    crps_ood = dfc["crps_ood"].to_numpy(float)
    labels_crps = dfc["method"].tolist()

    fig, axes = plt.subplots(1, 2, figsize=(14, max(4.5, 0.35 * len(labels_mae))))

    _grouped_barh(
        axes[0],
        labels_mae,
        mae_id,
        mae_ood,
        title=f"MAE: ID vs OOD ({'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'})",
        fmt=lambda v: f"{v:,.2f}",
    )

    _grouped_barh(
        axes[1],
        labels_crps,
        crps_id,
        crps_ood,
        title=f"Gaussian CRPS: ID vs OOD ({'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'})",
        fmt=lambda v: f"{v:,.2f}",
    )

    axes[0].legend(loc="lower right", frameon=False)
    axes[1].legend(loc="lower right", frameon=False)
    plt.tight_layout()
    savefig(fig, f"id_vs_ood_summary_{OOD_LABEL_TO_PLOT}_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split="test")
    plt.show()
else:
    print("[ID/OOD] No rows to plot.")


## Phase 7: Cohort sanity + aleatoric validity

Use dataset metadata to inspect cohort sizes and relate cohort error to predicted aleatoric uncertainty.

In [ ]:
# Phase 7A: cohort candidates quick check (uses config.cohorts)

import numpy as np
import pandas as pd

_coh = COHORTS if isinstance(COHORTS, dict) else {}
dataset_csv = _resolve_path(_coh.get("dataset_csv"))
if not _coh.get("enabled", False) or not dataset_csv or not dataset_csv.exists():
    print("[cohorts] disabled or dataset_csv missing; skipping.")
else:
    id_col = _coh.get("id_col", "id")
    cohort_defs = _coh.get("cohort_defs", {}) or {}
    age_col = _coh.get("age_col")
    age_bins = _coh.get("age_bins")
    age_from_cols = _coh.get("age_from_cols", []) or []
    min_n = int(_coh.get("min_cohort_n", 100))

    usecols = {id_col}
    for cols in cohort_defs.values():
        for c in cols:
            if c != "age_bucket":
                usecols.add(c)
    if age_col and age_col != "age_bucket" and not age_from_cols:
        usecols.add(age_col)
    for c in age_from_cols:
        usecols.add(c)

    # Only keep columns that actually exist in the CSV.
    header = pd.read_csv(dataset_csv, nrows=0).columns
    usecols = [c for c in usecols if c in header]

    df_meta = pd.read_csv(dataset_csv, usecols=usecols)

    if age_col and age_col not in df_meta.columns and len(age_from_cols) == 2:
        a0, a1 = age_from_cols
        if a0 in df_meta.columns and a1 in df_meta.columns:
            df_meta[age_col] = pd.to_numeric(df_meta[a0], errors="coerce") - pd.to_numeric(df_meta[a1], errors="coerce")

    if age_col and age_bins and age_col in df_meta.columns:
        df_meta[age_col] = pd.to_numeric(df_meta[age_col], errors="coerce")
        labels = [f"{age_bins[i]}-{age_bins[i+1]}" for i in range(len(age_bins) - 1)]
        df_meta["age_bucket"] = pd.cut(df_meta[age_col], bins=age_bins, labels=labels, include_lowest=True)

    rows = []
    for name, cols in cohort_defs.items():
        missing = [c for c in cols if c not in df_meta.columns]
        if missing:
            rows.append({"cohort_name": name, "cols": cols, "status": f"missing: {missing}"})
            continue
        tmp = df_meta.dropna(subset=cols).copy()
        key = tmp[cols].astype(str).agg(" / ".join, axis=1)
        counts = key.value_counts()
        rows.append({
            "cohort_name": name,
            "cols": " + \n".join(cols),
            "n_rows_used": int(counts.sum()),
            "n_cohorts": int(counts.size),
            "cohort_size_q50": float(np.quantile(counts.to_numpy(), 0.50)) if counts.size else np.nan,
            "cohort_size_q90": float(np.quantile(counts.to_numpy(), 0.90)) if counts.size else np.nan,
            "cohort_size_q99": float(np.quantile(counts.to_numpy(), 0.99)) if counts.size else np.nan,
            "n_cohorts_ge_min_n": int((counts >= min_n).sum()),
        })

    df_cohorts = pd.DataFrame(rows)
    EXPORTER.export_table(df_cohorts, "cohort_candidates", dataset=DATASET_KEY)
    display(df_cohorts)
    print(f"[cohorts] loaded {len(df_meta):,} rows from {dataset_csv}")


In [ ]:
# Phase 7B: noise-cohort analysis (aleatoric validity) on ID:test

import numpy as np
import pandas as pd

_coh = COHORTS if isinstance(COHORTS, dict) else {}
dataset_csv = _resolve_path(_coh.get("dataset_csv"))
if not _coh.get("enabled", False) or not dataset_csv or not dataset_csv.exists():
    print("[cohorts] disabled or dataset_csv missing; skipping.")
else:
    id_col = _coh.get("id_col", "id")
    cohort_defs = _coh.get("cohort_defs", {}) or {}
    age_col = _coh.get("age_col")
    age_bins = _coh.get("age_bins")
    min_n = int(_coh.get("min_cohort_n", 100))

    if "df_meta" not in globals():
        usecols = {id_col}
        for cols in cohort_defs.values():
            for c in cols:
                usecols.add(c)
        if age_col:
            usecols.add(age_col)
        df_meta = pd.read_csv(dataset_csv, usecols=list(usecols))
        if age_col and age_bins and age_col in df_meta.columns:
            df_meta[age_col] = pd.to_numeric(df_meta[age_col], errors="coerce")
            labels = [f"{age_bins[i]}-{age_bins[i+1]}" for i in range(len(age_bins) - 1)]
            df_meta["age_bucket"] = pd.cut(df_meta[age_col], bins=age_bins, labels=labels, include_lowest=True)

    if id_col != "id" and id_col in df_meta.columns:
        df_meta = df_meta.rename(columns={id_col: "id"})

    results = []
    for spec in METHODS:
        if spec.kind == "dido" or spec.head_type == "point":
            continue
        try:
            df_pred = method_id_frame(spec, "test")
        except Exception as e:
            print(f"[skip noise] {spec.key}: {type(e).__name__}: {e}")
            continue
        if "y_true" not in df_pred.columns or "mu" not in df_pred.columns:
            continue

        y = pd.to_numeric(df_pred["y_true"], errors="coerce").to_numpy(float)
        mu = pd.to_numeric(df_pred["mu"], errors="coerce").to_numpy(float)
        s_a = pd.to_numeric(df_pred.get("sigma_ale_raw"), errors="coerce").to_numpy(float)
        s_e = pd.to_numeric(df_pred.get("sigma_epi_raw"), errors="coerce").fillna(0.0).to_numpy(float)
        if USE_CALIBRATED_SIGMAS_LOCAL:
            s_a, _, s_t = calibrated_sigmas(
                head_type=spec.head_type,
                ale_src=spec.aleatoric_source,
                epi_src=spec.epistemic_source,
                sigma_ale_raw=s_a,
                sigma_epi_raw=s_e,
            )
        else:
            s_t = np.sqrt(np.maximum(np.nan_to_num(s_a, nan=0.0) ** 2 + np.nan_to_num(s_e, nan=0.0) ** 2, 0.0))

        df_small = pd.DataFrame({
            "id": pd.to_numeric(df_pred.get("id"), errors="coerce"),
            "abs_err": np.abs(y - mu),
            "sigma_ale": np.asarray(s_a, float),
            "sigma_tot": np.asarray(s_t, float),
        })
        df_join = df_small.merge(df_meta, on="id", how="inner")

        for name, cols in cohort_defs.items():
            if any(c not in df_join.columns for c in cols):
                continue
            tmp = df_join.dropna(subset=cols).copy()
            g = tmp.groupby(cols).agg(
                n=("id", "size"),
                mean_abs_err=("abs_err", "mean"),
                mean_sigma_ale=("sigma_ale", "mean"),
                mean_sigma_tot=("sigma_tot", "mean"),
            )
            g = g[g["n"] >= min_n]
            if g.empty:
                continue
            corr_ale = float(g["mean_sigma_ale"].corr(g["mean_abs_err"]))
            corr_tot = float(g["mean_sigma_tot"].corr(g["mean_abs_err"]))
            bias_ale = float((g["mean_sigma_ale"] - g["mean_abs_err"]).mean())
            bias_tot = float((g["mean_sigma_tot"] - g["mean_abs_err"]).mean())

            results.append({
                "method": spec.label,
                "key": spec.key,
                "cohort": name,
                "n_cohorts": int(len(g)),
                "corr_sigma_ale_vs_err": corr_ale,
                "corr_sigma_tot_vs_err": corr_tot,
                "bias_sigma_ale_minus_err": bias_ale,
                "bias_sigma_tot_minus_err": bias_tot,
            })

    df_noise = pd.DataFrame(results)
    EXPORTER.export_table(df_noise, "noise_cohort_stats", dataset=DATASET_KEY, split="test")
    display(df_noise.sort_values(["cohort", "corr_sigma_ale_vs_err"], ascending=[True, False]))


In [ ]:
# Phase 7C: noise-cohort plots (log-log)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

USE_CALIBRATED_SIGMAS_LOCAL = USE_CALIBRATED_SIGMAS if "USE_CALIBRATED_SIGMAS" in globals() else True

_coh = COHORTS if isinstance(COHORTS, dict) else {}
dataset_csv = _resolve_path(_coh.get("dataset_csv"))
if not _coh.get("enabled", False) or not dataset_csv or not dataset_csv.exists():
    print("[cohorts] disabled or dataset_csv missing; skipping.")
else:
    id_col = _coh.get("id_col", "id")
    cohort_defs = _coh.get("cohort_defs", {}) or {}
    age_col = _coh.get("age_col")
    age_bins = _coh.get("age_bins")
    min_n = int(_coh.get("min_cohort_n", 100))

    if "df_meta" not in globals():
        usecols = {id_col}
        for cols in cohort_defs.values():
            for c in cols:
                usecols.add(c)
        if age_col:
            usecols.add(age_col)
        df_meta = pd.read_csv(dataset_csv, usecols=list(usecols))
        if age_col and age_bins and age_col in df_meta.columns:
            df_meta[age_col] = pd.to_numeric(df_meta[age_col], errors="coerce")
            labels = [f"{age_bins[i]}-{age_bins[i+1]}" for i in range(len(age_bins) - 1)]
            df_meta["age_bucket"] = pd.cut(df_meta[age_col], bins=age_bins, labels=labels, include_lowest=True)

    if id_col != "id" and id_col in df_meta.columns:
        df_meta = df_meta.rename(columns={id_col: "id"})

    def _attach_meta(df_eval: pd.DataFrame, cols_needed: list[str]) -> pd.DataFrame:
        if "id" not in df_eval.columns:
            raise KeyError("Eval frame missing 'id' column; cannot join cohorts")
        out = df_eval.copy()
        out["id"] = pd.to_numeric(out["id"], errors="coerce")
        out = out.dropna(subset=["id"])
        out["id"] = out["id"].astype(int)
        meta_cols = ["id"] + [c for c in cols_needed if c in df_meta.columns]
        out = out.merge(df_meta[meta_cols], on="id", how="left")
        return out

    def _cohort_key(df: pd.DataFrame, cols: list[str]) -> pd.Series:
        return df[cols].astype(str).agg(" / ".join, axis=1)

    def _aleatoric_sigma(spec: MethodSpec, df_eval: pd.DataFrame) -> np.ndarray:
        sA_raw = pd.to_numeric(df_eval.get("sigma_ale_raw"), errors="coerce").to_numpy(float)
        if spec.kind == "nf":
            return np.asarray(sA_raw, float)
        if USE_CALIBRATED_SIGMAS_LOCAL:
            sE_zeros = np.zeros_like(sA_raw, dtype=float)
            sA, _, _ = calibrated_sigmas(
                head_type=spec.head_type,
                ale_src=spec.aleatoric_source,
                epi_src="none",
                sigma_ale_raw=sA_raw,
                sigma_epi_raw=sE_zeros,
            )
            return np.asarray(sA, float)
        return np.asarray(sA_raw, float)

    # Select analytic Laplace/Gauss + NF variants (if present).
    specs = []
    for m in METHODS:
        if m.head_type == "point" or m.kind == "dido":
            continue
        if m.head_type in {"gauss", "laplace"} and str(m.epistemic_source).lower() == "none" and str(m.aleatoric_source).lower() == "analytic":
            specs.append(m)
        if m.kind == "nf":
            specs.append(m)

    seen = set()
    specs = [m for m in specs if (m.label not in seen and not seen.add(m.label))]

    tables = []
    for spec in specs:
        try:
            df_id = method_id_frame(spec, "test")
        except Exception as e:
            print(f"[skip noise plot] {spec.key}: {type(e).__name__}: {e}")
            continue
        if "y_true" not in df_id.columns or "mu" not in df_id.columns:
            continue

        y = pd.to_numeric(df_id["y_true"], errors="coerce").to_numpy(float)
        mu = pd.to_numeric(df_id["mu"], errors="coerce").to_numpy(float)
        resid = y - mu
        sA = _aleatoric_sigma(spec, df_id)

        for cohort_name, cols in cohort_defs.items():
            needed_cols = list(cols)
            df_join = _attach_meta(df_id, needed_cols)
            if any(c not in df_join.columns for c in cols):
                continue
            tmp = pd.DataFrame(
                {"cohort": _cohort_key(df_join, cols), "resid": resid, "sA2": np.square(sA)}
            )
            tmp = tmp[np.isfinite(tmp["resid"]) & np.isfinite(tmp["sA2"])].copy()
            g = (
                tmp.groupby("cohort")
                .agg(n=("resid", "size"), resid_var=("resid", "var"), mean_sA2=("sA2", "mean"))
                .reset_index()
            )
            g = g[g["n"] >= min_n].copy()
            if g.empty:
                continue
            g["cohort_def"] = cohort_name
            g["method"] = spec.label
            tables.append(g)

    if not tables:
        print("[noise plot] No cohort tables produced.")
    else:
        df_noise = pd.concat(tables, ignore_index=True)
        EXPORTER.export_table(df_noise, f"noise_cohort_scatter_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split="test")

        def _plot_grid_for_cohort(cohort_def: str, df_all: pd.DataFrame):
            methods_here = sorted(df_all["method"].unique().tolist())
            n = len(methods_here)
            if n == 0:
                return
            ncols = 2
            nrows = int(np.ceil(n / ncols))
            fig, axes = plt.subplots(nrows, ncols, figsize=(6.5 * ncols, 5.2 * nrows), constrained_layout=True)
            axes = np.atleast_1d(axes).reshape(-1)

            for ax, method in zip(axes, methods_here):
                sub = df_all[(df_all["method"] == method) & (df_all["resid_var"] > 0) & (df_all["mean_sA2"] > 0)].copy()
                if sub.empty:
                    ax.set_axis_off()
                    continue
                ax.scatter(sub["resid_var"], sub["mean_sA2"], s=40, alpha=0.65)
                ax.set_xscale("log")
                ax.set_yscale("log")
                ax.set_xlabel("Var(residual) within cohort")
                ax.set_ylabel("E[sigma_a^2] within cohort")
                ax.set_title(f"{method} (n_cohorts={len(sub)})")
                ax.grid(True, which="both", alpha=0.2)

            for ax in axes[n:]:
                ax.set_axis_off()

            fig.suptitle(
                f"Noise-cohort (ID:test): {cohort_def} — {'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}",
                y=1.02,
            )
            savefig(fig, f"noise_cohort_{cohort_def}_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split="test")
            plt.show()

        for cohort_def, sub in df_noise.groupby("cohort_def"):
            _plot_grid_for_cohort(cohort_def, sub)


## Phase 8: Uncertainty behavior and decision analysis

Sharpness, large-error AURC, decile diagnostics, selective prediction, and blend synergy.

In [ ]:
# Phase 8A: shared helpers

import numpy as np
import pandas as pd

USE_CALIBRATED_SIGMAS_LOCAL = USE_CALIBRATED_SIGMAS if "USE_CALIBRATED_SIGMAS" in globals() else True
EVAL_SCOPE = "id_test"  # "id_test" | "ood" | "id_test+ood"
OOD_LABELS_LOCAL = list(OOD_LABELS) if "OOD_LABELS" in globals() else []

SIG_COLORS = {"Ua": "tab:blue", "Ue": "tab:green", "Utot": "tab:orange", "blend": "tab:purple", "random": "0.5"}


def _get_series(df: pd.DataFrame, col: str) -> pd.Series:
    if col in df.columns:
        return pd.to_numeric(df[col], errors="coerce")
    return pd.Series(np.nan, index=df.index)


def _gather_eval_frames(spec: MethodSpec) -> pd.DataFrame:
    parts = []
    scope = EVAL_SCOPE.strip().lower()

    if scope in {"id_test", "id_test+ood"}:
        df_id = method_id_frame(spec, "test").copy()
        df_id["dataset"] = "id:test"
        parts.append(df_id)

    if scope in {"ood", "id_test+ood"}:
        for ood_label in OOD_LABELS_LOCAL:
            df_o = method_ood_frame(spec, ood_label).copy()
            df_o["dataset"] = f"ood:{ood_label}"
            parts.append(df_o)

    if not parts:
        raise ValueError(f"Unknown EVAL_SCOPE={EVAL_SCOPE!r}")
    return pd.concat(parts, ignore_index=True)


def _sigmas(spec: MethodSpec, df: pd.DataFrame):
    s_a_raw = _get_series(df, "sigma_ale_raw").to_numpy(float)
    s_e_raw = _get_series(df, "sigma_epi_raw").to_numpy(float)

    if USE_CALIBRATED_SIGMAS_LOCAL:
        s_a, s_e, s_t = calibrated_sigmas(
            head_type=spec.head_type,
            ale_src=spec.aleatoric_source,
            epi_src=spec.epistemic_source,
            sigma_ale_raw=s_a_raw,
            sigma_epi_raw=np.nan_to_num(s_e_raw, nan=0.0),
        )
    else:
        s_a = np.asarray(s_a_raw, float)
        s_e = np.asarray(s_e_raw, float)
        s_t = np.sqrt(np.maximum(np.nan_to_num(s_a, nan=0.0) ** 2 + np.nan_to_num(s_e, nan=0.0) ** 2, 0.0))

    s_a = np.asarray(s_a, float)
    s_e = np.asarray(s_e, float)
    s_t = np.asarray(s_t, float)

    s_e[~np.isfinite(s_e_raw)] = np.nan
    s_t[~(np.isfinite(s_a_raw) | np.isfinite(s_e_raw))] = np.nan
    return s_a, s_e, s_t


SPECS_SIGMA = [s for s in METHODS if s.head_type != "point" and s.kind != "dido"]
SPECS_EPI = [s for s in SPECS_SIGMA if str(s.epistemic_source).lower() in {"mc", "ensemble"}]

print(f"[phase8] scope={EVAL_SCOPE} calibrated={USE_CALIBRATED_SIGMAS_LOCAL} specs_sigma={len(SPECS_SIGMA)} specs_epi={len(SPECS_EPI)}")


In [ ]:
# Phase 8B: sharpness (interval width vs nominal coverage)

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

LEVELS = np.arange(0.10, 1.00, 0.10)
LEVEL_90 = 0.90

def _z_for_level(level: float) -> float:
    return float(norm.ppf((1.0 + float(level)) / 2.0))

Z = {lvl: _z_for_level(lvl) for lvl in LEVELS}
Z90 = _z_for_level(LEVEL_90)

rows = []
width90_rows = []
curve_cache = {}

for spec in SPECS_SIGMA:
    try:
        df_all = _gather_eval_frames(spec)
    except Exception as e:
        print(f"[skip sharpness] {spec.key}: {type(e).__name__}: {e}")
        continue

    _, _, s_t = _sigmas(spec, df_all)
    mask = np.isfinite(s_t) & (s_t > 0)
    if not np.any(mask):
        print(f"[skip sharpness] {spec.key}: no finite sigma_total")
        continue
    s_t = s_t[mask]

    mean_widths = []
    for lvl in LEVELS:
        w = 2.0 * Z[lvl] * s_t
        mean_widths.append(float(np.mean(w)))
        rows.append({
            "method": spec.label,
            "key": spec.key,
            "eval_scope": EVAL_SCOPE,
            "nominal": float(lvl),
            "mean_width": float(np.mean(w)),
        })

    curve_cache[spec.label] = (LEVELS.copy(), np.array(mean_widths, float))
    w90 = float(np.mean(2.0 * Z90 * s_t))
    width90_rows.append({
        "method": spec.label,
        "key": spec.key,
        "eval_scope": EVAL_SCOPE,
        "nominal": LEVEL_90,
        "mean_width": w90,
    })

df_width_90 = pd.DataFrame(width90_rows).sort_values("mean_width")
EXPORTER.export_table(df_width_90, f"sharpness_width90_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split=EVAL_SCOPE)
display(df_width_90)

# subplot grid for curves
if curve_cache:
    methods = list(curve_cache.keys())
    n = len(methods)
    ncols = 2
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4.2 * nrows), sharex=True, sharey=True, constrained_layout=True)
    axes = np.atleast_1d(axes).reshape(-1)

    for ax, method in zip(axes, methods):
        lvls, mws = curve_cache[method]
        ax.plot(lvls * 100, mws, marker="o", linewidth=2, alpha=0.9)
        ax.set_title(method)
        ax.set_xlabel("Nominal coverage (%)")
        ax.set_ylabel("Mean interval width (2*z*sigma_tot)")
        ax.grid(True, alpha=0.2)

    for ax in axes[n:]:
        ax.set_axis_off()

    fig.suptitle(f"Sharpness: mean width vs coverage ({EVAL_SCOPE}, {'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'})")
    savefig(fig, f"sharpness_width_curves_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split=EVAL_SCOPE)
    plt.show()

if not df_width_90.empty:
    plt.figure(figsize=(8, max(3.5, 0.45 * len(df_width_90))))
    y = np.arange(len(df_width_90))
    vals = df_width_90["mean_width"].to_numpy(float)
    plt.barh(y, vals, alpha=0.85)
    plt.yticks(y, df_width_90["method"].tolist())
    plt.gca().invert_yaxis()
    plt.xlabel("Mean width at 90% nominal coverage")
    plt.title(f"Sharpness @90% ({EVAL_SCOPE}, {'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'})")
    plt.grid(True, axis="x", alpha=0.2)
    plt.gca().ticklabel_format(style="plain", axis="x", useOffset=False)
    xmax = float(vals.max()) if len(vals) else 0.0
    pad = 0.02 * xmax
    for yi, v in zip(y, vals):
        plt.text(max(v - pad, 0.0), yi, f"{v:.2f}", va="center", ha="right", fontsize=9)
    plt.tight_layout()
    savefig(plt.gcf(), f"sharpness_width90_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split=EVAL_SCOPE)
    plt.show()


In [ ]:
# Phase 5A3: OOD detection AUROC/AUPRC + gains curves

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score

USE_CALIBRATED_SIGMAS_LOCAL = USE_CALIBRATED_SIGMAS if "USE_CALIBRATED_SIGMAS" in globals() else True
OOD_LABELS_LOCAL = list(OOD_LABELS) if "OOD_LABELS" in globals() else []
SELECT_SIGNALS = ["Ua", "Ue", "Utot"]
SIG_COLORS_LOCAL = (
    SIG_COLORS
    if "SIG_COLORS" in globals()
    else {"Ua": "tab:blue", "Ue": "tab:green", "Utot": "tab:orange", "blend": "tab:purple", "random": "0.5"}
)

INCLUDE_DIDO = True
DIDO_SIGNAL = settings.get("dido_signal", "dido_strength_raw")
DIDO_CANDIDATES = ["dido_entropy_raw", "dido_vacuity_raw", "dido_strength_raw"]
DIDO_ALLOW_FLIP = True


def _label_candidates(label: str) -> list[str]:
    c = [label]
    if label.endswith("_hard"):
        c.append(label.replace("_hard", ""))
    return list(dict.fromkeys(c))


def _ood_metrics(y_true_bin: np.ndarray, score: np.ndarray):
    y_true_bin = np.asarray(y_true_bin, int)
    score = np.asarray(score, float)
    mask = np.isfinite(score) & np.isfinite(y_true_bin)
    if not np.any(mask):
        return float("nan"), float("nan"), 0
    y = y_true_bin[mask]
    s = score[mask]
    if len(np.unique(y)) < 2:
        return float("nan"), float("nan"), int(mask.sum())
    return float(roc_auc_score(y, s)), float(average_precision_score(y, s)), int(mask.sum())


def _gains_curve(y_ood: np.ndarray, score: np.ndarray):
    y_ood = np.asarray(y_ood, int)
    score = np.asarray(score, float)
    m = np.isfinite(score) & np.isfinite(y_ood)
    if not np.any(m):
        return None
    y = y_ood[m]
    s = score[m]
    if len(np.unique(y)) < 2:
        return None
    order = np.argsort(-s, kind="mergesort")
    y = y[order]
    total_pos = float(np.sum(y))
    if total_pos <= 0:
        return None
    cum_pos = np.cumsum(y).astype(float)
    recall = cum_pos / total_pos
    coverage = (np.arange(1, len(y) + 1) / float(len(y))).astype(float)
    aurc = float(np.trapezoid(recall, coverage))
    return coverage, recall, aurc


def _sigmas(spec: MethodSpec, df: pd.DataFrame):
    s_a_raw = pd.to_numeric(df.get("sigma_ale_raw"), errors="coerce").to_numpy(float)
    s_e_raw = pd.to_numeric(df.get("sigma_epi_raw"), errors="coerce").to_numpy(float)
    miss_epi = ~np.isfinite(s_e_raw)
    s_e0 = np.nan_to_num(s_e_raw, nan=0.0)

    if USE_CALIBRATED_SIGMAS_LOCAL:
        s_a, s_e, s_t = calibrated_sigmas(
            head_type=spec.head_type,
            ale_src=spec.aleatoric_source,
            epi_src=spec.epistemic_source,
            sigma_ale_raw=s_a_raw,
            sigma_epi_raw=s_e0,
        )
        s_e = np.asarray(s_e, float)
        s_e[miss_epi] = np.nan
        return np.asarray(s_a, float), s_e, np.asarray(s_t, float)

    s_a = np.asarray(s_a_raw, float)
    s_e = np.asarray(s_e_raw, float)
    s_t = np.sqrt(np.maximum(np.nan_to_num(s_a, nan=0.0) ** 2 + np.nan_to_num(s_e, nan=0.0) ** 2, 0.0))
    return s_a, s_e, s_t


def _load_ood_for_spec(spec: MethodSpec, ood_label: str) -> pd.DataFrame:
    last_err = None
    for lab in _label_candidates(ood_label):
        try:
            return method_ood_frame(spec, lab)
        except Exception as e:
            last_err = e
    raise last_err


base_specs = [m for m in METHODS if m.kind in {"regression", "ensemble"} and str(m.epistemic_source).lower() in {"mc", "ensemble"}]
rows = []
gains_rows = []

for ood_label in OOD_LABELS_LOCAL:
    for spec in base_specs:
        try:
            df_id = method_id_frame(spec, "test")
            df_ood = _load_ood_for_spec(spec, ood_label)
        except Exception as e:
            print(f"[skip] {spec.label} / ood={ood_label}: {type(e).__name__}: {e}")
            continue

        df_id = df_id.copy()
        df_ood = df_ood.copy()
        df_id["is_ood"] = 0
        df_ood["is_ood"] = 1
        df_all = pd.concat([df_id, df_ood], ignore_index=True)
        ybin = df_all["is_ood"].to_numpy(int)

        s_a, s_e, s_t = _sigmas(spec, df_all)
        sig_map = {"Ua": s_a, "Ue": s_e, "Utot": s_t}

        for sig_name in SELECT_SIGNALS:
            score = sig_map[sig_name]
            auroc, auprc, n_used = _ood_metrics(ybin, score)
            rows.append({
                "ood_label": ood_label,
                "method": spec.label,
                "signal": sig_name,
                "n_id": int(len(df_id)),
                "n_ood": int(len(df_ood)),
                "n_used": n_used,
                "auroc": auroc,
                "auprc": auprc,
                "use_calibrated": bool(USE_CALIBRATED_SIGMAS_LOCAL),
            })

            g = _gains_curve(ybin, score)
            if g is not None:
                cov, rec, aurc = g
                gains_rows.append({
                    "ood_label": ood_label,
                    "method": spec.label,
                    "signal": sig_name,
                    "coverage": cov,
                    "recall": rec,
                    "aurc": aurc,
                })

    dido_spec = next((m for m in METHODS if m.kind == "dido" and m.key == "dido_lpl"), None)
    if INCLUDE_DIDO and dido_spec is not None:
        try:
            df_id_d = method_id_frame(dido_spec, "test")
            df_ood_d = _load_ood_for_spec(dido_spec, ood_label)
        except Exception as e:
            print(f"[skip] Laplace DIDO / ood={ood_label}: {type(e).__name__}: {e}")
        else:
            df_id_d = df_id_d.copy()
            df_ood_d = df_ood_d.copy()
            df_id_d["is_ood"] = 0
            df_ood_d["is_ood"] = 1
            df_all_d = pd.concat([df_id_d, df_ood_d], ignore_index=True)
            ybin = df_all_d["is_ood"].to_numpy(int)

            if str(DIDO_SIGNAL).lower() == "auto":
                dido_cols = [c for c in DIDO_CANDIDATES if c in df_all_d.columns]
            else:
                dido_cols = [str(DIDO_SIGNAL)] if str(DIDO_SIGNAL) in df_all_d.columns else []

            if not dido_cols:
                print(f"[skip] Laplace DIDO / ood={ood_label}: dido column not found")
            else:
                best = None
                for col in dido_cols:
                    score0 = pd.to_numeric(df_all_d[col], errors="coerce").to_numpy(float)
                    auroc, auprc, n_used = _ood_metrics(ybin, score0)
                    flipped = False
                    score_used = score0
                    if DIDO_ALLOW_FLIP:
                        auroc_neg, auprc_neg, _ = _ood_metrics(ybin, -score0)
                        if np.isfinite(auroc_neg) and (not np.isfinite(auroc) or auroc_neg > auroc):
                            auroc, auprc, flipped = auroc_neg, auprc_neg, True
                            score_used = -score0

                    cand = (float(auroc), float(auprc), col, flipped, int(n_used), score_used)
                    if best is None or (np.isfinite(cand[0]) and (not np.isfinite(best[0]) or cand[0] > best[0])):
                        best = cand

                if best is not None:
                    auroc, auprc, col, flipped, n_used, score_used = best
                    sig_name = col + (" (flipped)" if flipped else "")
                    rows.append({
                        "ood_label": ood_label,
                        "method": "Laplace + DIDO",
                        "signal": sig_name,
                        "n_id": int(len(df_id_d)),
                        "n_ood": int(len(df_ood_d)),
                        "n_used": int(n_used),
                        "auroc": auroc,
                        "auprc": auprc,
                        "use_calibrated": False,
                    })
                    g = _gains_curve(ybin, score_used)
                    if g is not None:
                        cov, rec, aurc = g
                        gains_rows.append({
                            "ood_label": ood_label,
                            "method": "Laplace + DIDO",
                            "signal": sig_name,
                            "coverage": cov,
                            "recall": rec,
                            "aurc": aurc,
                        })

if rows:
    df_ooddet = pd.DataFrame(rows)
    EXPORTER.export_table(df_ooddet, f"ood_detection_full_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split="test")
    display(df_ooddet.sort_values(["ood_label", "auroc"], ascending=[True, False]).reset_index(drop=True))

    for ood_label in OOD_LABELS_LOCAL:
        sub = df_ooddet[df_ooddet["ood_label"] == ood_label].copy()
        sub = sub[np.isfinite(sub["auroc"].to_numpy(float))].copy()
        if sub.empty:
            continue
        sub["name"] = sub["method"] + " / " + sub["signal"]
        sub = sub.sort_values("auroc", ascending=True)

        fig, ax = plt.subplots(figsize=(9, max(3.5, 0.35 * len(sub))))
        y = np.arange(len(sub))
        vals = sub["auroc"].to_numpy(float)
        ax.barh(y, vals, alpha=0.85)
        ax.set_yticks(y)
        ax.set_yticklabels(sub["name"].tolist())
        ax.set_xlabel("AUROC (ID vs OOD)")
        ax.set_title(
            f"OOD detection AUROC - {ood_label} ({'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'})"
            + f"  signals={','.join(SELECT_SIGNALS)}"
        )
        ax.grid(True, axis="x", alpha=0.2)
        ax.ticklabel_format(style="plain", axis="x", useOffset=False)
        xmax = float(vals.max()) if len(vals) else 0.0
        pad = 0.02 * xmax
        for yi, v in zip(y, vals):
            ax.text(max(v - pad, 0.0), yi, f"{v:.3f}", va="center", ha="right", fontsize=9)
        fig.tight_layout()
        fig.subplots_adjust(left=0.45)
        savefig(fig, f"ood_detection_auroc_{ood_label}_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split="test")
        plt.show()

    gains_map = {(r["ood_label"], r["method"], r["signal"]): (r["coverage"], r["recall"], r["aurc"]) for r in gains_rows}

    for ood_label in OOD_LABELS_LOCAL:
        methods = sorted({m for (ol, m, _s) in gains_map.keys() if ol == ood_label})
        if not methods:
            continue

        ncols = 2
        nrows = int(np.ceil(len(methods) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(7.5 * ncols, 5.0 * nrows), sharex=False, sharey=False, constrained_layout=True)
        axes = np.atleast_1d(axes).reshape(-1)

        for ax, method in zip(axes, methods):
            ax.plot([0, 1], [0, 1], "--", alpha=0.35, color=SIG_COLORS_LOCAL.get("random", "0.5"), label="random")

            sigs_here = [sig for (ol, m, sig) in gains_map.keys() if (ol == ood_label and m == method)]
            for sig in sigs_here:
                cov, rec, aurc = gains_map[(ood_label, method, sig)]
                color = SIG_COLORS_LOCAL.get(sig, None) if sig in {"Ua", "Ue", "Utot"} else None
                ax.plot(cov, rec, lw=2, color=color, label=f"{sig} (AURC={aurc:.3f})")

            ax.set_title(method)
            ax.set_xlabel("Coverage reviewed (fraction, highest score first)")
            ax.set_ylabel("Recall of OOD")
            ax.set_ylim(0, 1.05)
            ax.grid(True, alpha=0.2)
            ax.legend(frameon=False)

        for ax in axes[len(methods):]:
            ax.set_axis_off()

        fig.suptitle(
            f"OOD gains curves - {ood_label} ({'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'})",
            y=1.02,
        )
        savefig(fig, f"ood_gains_curves_{ood_label}_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split="test")
        plt.show()


In [ ]:
# Phase 8D: big-error share by uncertainty decile (Ua/Ue/Utot)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    TAU = float(TAU)
except Exception as e:
    raise RuntimeError("TAU not defined; run the large-error cell first.") from e

N_BINS = 10
SIGNALS = ["Ua", "Ue", "Utot"]

if not SPECS_EPI:
    print("[deciles] No epistemic methods available.")
else:
    ncols = 2
    nrows = int(np.ceil(len(SPECS_EPI) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 3.8 * nrows), sharex=False, sharey=False)
    axes = np.array(axes).reshape(-1)

    for ax, spec in zip(axes, SPECS_EPI):
        try:
            df_all = _gather_eval_frames(spec)
        except Exception as e:
            ax.set_axis_off()
            ax.set_title(f"{spec.label} (skip: {type(e).__name__})")
            continue

        if "y_true" not in df_all.columns or "mu" not in df_all.columns:
            ax.set_axis_off()
            ax.set_title(f"{spec.label} (missing y_true/mu)")
            continue

        y = pd.to_numeric(df_all["y_true"], errors="coerce").to_numpy(float)
        mu = pd.to_numeric(df_all["mu"], errors="coerce").to_numpy(float)
        err = np.abs(y - mu)
        is_large = err > TAU
        s_a, s_e, s_t = _sigmas(spec, df_all)
        sig_map = {"Ua": s_a, "Ue": s_e, "Utot": s_t}

        for sig_name in SIGNALS:
            scores = sig_map[sig_name]
            mask = np.isfinite(scores) & np.isfinite(err)
            if mask.sum() < 10:
                continue
            try:
                bins = pd.qcut(scores[mask], q=N_BINS, labels=False, duplicates="drop")
            except ValueError:
                continue
            max_bin = int(bins.max())
            shares = []
            for b in range(max_bin + 1):
                idx = (bins == b)
                share = float(np.mean(is_large[mask][idx])) if np.any(idx) else np.nan
                shares.append(share)
            ax.plot(range(1, len(shares) + 1), shares, marker="o", label=sig_name, color=SIG_COLORS.get(sig_name, None))

        ax.set_title(spec.label)
        ax.set_xlabel("Uncertainty decile (low -> high)")
        ax.set_ylabel("Share of large errors")
        ax.grid(alpha=0.2)
        ax.legend(frameon=False)

    for ax in axes[len(SPECS_EPI):]:
        ax.set_axis_off()

    fig.suptitle(f"Large-error share by uncertainty decile ({EVAL_SCOPE})")
    fig.tight_layout(rect=[0, 0.03, 1, 0.98])
    savefig(fig, f"large_error_share_by_decile_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split=EVAL_SCOPE)
    plt.show()


In [ ]:
# Phase 8E: epistemic selective prediction (risk vs coverage)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

COVERAGES = np.linspace(0.1, 1.0, 10)

if not SPECS_EPI:
    print("[selective] No epistemic methods available.")
else:
    plt.figure(figsize=(8, 5))
    for spec in SPECS_EPI:
        try:
            df_all = _gather_eval_frames(spec)
        except Exception as e:
            print(f"[skip selective] {spec.key}: {type(e).__name__}: {e}")
            continue
        if "y_true" not in df_all.columns or "mu" not in df_all.columns:
            continue

        y = pd.to_numeric(df_all["y_true"], errors="coerce").to_numpy(float)
        mu = pd.to_numeric(df_all["mu"], errors="coerce").to_numpy(float)
        _, s_e, _ = _sigmas(spec, df_all)

        mask = np.isfinite(y) & np.isfinite(mu) & np.isfinite(s_e)
        if mask.sum() == 0:
            continue

        y = y[mask]
        mu = mu[mask]
        s_e = s_e[mask]

        order = np.argsort(s_e)  # low epistemic = keep
        y = y[order]
        mu = mu[order]

        maes = []
        for cov in COVERAGES:
            k = max(1, int(np.floor(cov * len(y))))
            maes.append(float(np.mean(np.abs(y[:k] - mu[:k]))))

        plt.plot(COVERAGES, maes, marker="o", label=spec.label)

    plt.xlabel("Coverage (fraction kept; low Ue)")
    plt.ylabel("MAE on kept subset")
    plt.title(f"Selective prediction via Ue ({EVAL_SCOPE})")
    plt.grid(alpha=0.2)
    plt.legend(fontsize=8)
    plt.tight_layout()
    savefig(plt.gcf(), f"selective_prediction_ue_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split=EVAL_SCOPE)
    plt.show()


In [ ]:
# Phase 8F: decision synergy (ALE vs EPI vs BLEND)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

WEIGHTS = np.linspace(0.0, 1.0, 21)
COLOR_ALE = "tab:blue"
COLOR_EPI = "tab:orange"
COLOR_BLEND = "tab:green"

try:
    TAU = float(TAU)
except Exception as e:
    raise RuntimeError("TAU not defined; run the large-error cell first.") from e

def _aurc_from_scores(scores: np.ndarray, labels: np.ndarray) -> float:
    order = np.argsort(scores)[::-1]
    y = labels[order].astype(float)
    if y.sum() == 0:
        return float("nan")
    recall = np.cumsum(y) / y.sum()
    coverage = (np.arange(len(y)) + 1) / len(y)
    return float(np.trapezoid(recall, coverage))

if not SPECS_EPI:
    print("[synergy] No epistemic methods available.")
else:
    ncols = 2
    nrows = int(np.ceil(len(SPECS_EPI) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(7 * ncols, 4 * nrows), sharex=False, sharey=False)
    axes = np.array(axes).reshape(-1)

    for ax, spec in zip(axes, SPECS_EPI):
        try:
            df_all = _gather_eval_frames(spec)
        except Exception as e:
            ax.set_axis_off()
            ax.set_title(f"{spec.label} (skip: {type(e).__name__})")
            continue

        if "y_true" not in df_all.columns or "mu" not in df_all.columns:
            ax.set_axis_off()
            ax.set_title(f"{spec.label} (missing y_true/mu)")
            continue

        y = pd.to_numeric(df_all["y_true"], errors="coerce").to_numpy(float)
        mu = pd.to_numeric(df_all["mu"], errors="coerce").to_numpy(float)
        err = np.abs(y - mu)
        is_large = err > TAU

        s_a, s_e, _ = _sigmas(spec, df_all)
        mask = np.isfinite(s_a) & np.isfinite(s_e) & np.isfinite(err)
        if mask.sum() == 0 or is_large[mask].sum() == 0:
            ax.set_axis_off()
            ax.set_title(f"{spec.label} (no large errors)")
            continue

        s_a = s_a[mask]
        s_e = s_e[mask]
        y_large = is_large[mask]

        aurc_ale = _aurc_from_scores(s_a, y_large)
        aurc_epi = _aurc_from_scores(s_e, y_large)

        best_w, best_aurc = 0.0, -np.inf
        best_scores = None
        for w in WEIGHTS:
            blend = w * s_e + (1.0 - w) * s_a
            aurc = _aurc_from_scores(blend, y_large)
            if np.isfinite(aurc) and aurc > best_aurc:
                best_aurc = aurc
                best_w = w
                best_scores = blend

        def _plot_curve(scores, label, color):
            order = np.argsort(scores)[::-1]
            y = y_large[order].astype(float)
            recall = np.cumsum(y) / y.sum()
            coverage = (np.arange(len(y)) + 1) / len(y)
            ax.plot(coverage, recall, label=label, color=color)

        _plot_curve(s_a, f"Ua (AURC={aurc_ale:.3f})", COLOR_ALE)
        _plot_curve(s_e, f"Ue (AURC={aurc_epi:.3f})", COLOR_EPI)
        if best_scores is not None:
            synergy = best_aurc - max(aurc_ale, aurc_epi)
            _plot_curve(best_scores, f"blend w={best_w:.2f} (AURC={best_aurc:.3f}, +{synergy:.3f})", COLOR_BLEND)

        ax.set_title(spec.label)
        ax.set_xlabel("Coverage (sorted by score)")
        ax.set_ylabel("Large-error recall")
        ax.grid(alpha=0.2)
        ax.legend(fontsize=7, frameon=False)

    for ax in axes[len(SPECS_EPI):]:
        ax.set_axis_off()

    fig.suptitle(f"Decision synergy (tau={TAU:.3g}, {EVAL_SCOPE})")
    fig.tight_layout(rect=[0, 0.03, 1, 0.98])
    savefig(fig, f"decision_synergy_blend_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split=EVAL_SCOPE)
    plt.show()


In [ ]:
# Phase 8G: unique information checks (Ua vs Ue)

import numpy as np
import pandas as pd

N_BINS = 10
EPS = 1e-8

try:
    TAU = float(TAU)
except Exception as e:
    raise RuntimeError("TAU not defined; run the large-error cell first.") from e

rows = []
for spec in SPECS_EPI:
    try:
        df_all = _gather_eval_frames(spec)
    except Exception as e:
        print(f"[skip unique] {spec.key}: {type(e).__name__}: {e}")
        continue
    if "y_true" not in df_all.columns or "mu" not in df_all.columns:
        continue

    y = pd.to_numeric(df_all["y_true"], errors="coerce").to_numpy(float)
    mu = pd.to_numeric(df_all["mu"], errors="coerce").to_numpy(float)
    err = np.abs(y - mu)
    s_a, s_e, _ = _sigmas(spec, df_all)
    mask = np.isfinite(err) & np.isfinite(s_a) & np.isfinite(s_e)
    if mask.sum() < 100:
        continue

    err = err[mask]
    s_a = s_a[mask]
    s_e = s_e[mask]

    z = err / (s_a + EPS)

    # Within Ua bins: does z rise with Ue?
    corrs = []
    try:
        bins_ua = pd.qcut(s_a, q=N_BINS, labels=False, duplicates="drop")
        for b in range(int(bins_ua.max()) + 1):
            idx = bins_ua == b
            if idx.sum() < 20:
                continue
            corr = np.corrcoef(z[idx], s_e[idx])[0, 1]
            if np.isfinite(corr):
                corrs.append(corr)
    except ValueError:
        pass
    corr_z_ue = float(np.nanmean(corrs)) if corrs else np.nan

    # Within Ue bins: does |err| rise with Ua?
    corr2 = []
    try:
        bins_ue = pd.qcut(s_e, q=N_BINS, labels=False, duplicates="drop")
        for b in range(int(bins_ue.max()) + 1):
            idx = bins_ue == b
            if idx.sum() < 20:
                continue
            c = np.corrcoef(err[idx], s_a[idx])[0, 1]
            if np.isfinite(c):
                corr2.append(c)
    except ValueError:
        pass
    corr_err_ua = float(np.nanmean(corr2)) if corr2 else np.nan

    rows.append({
        "method": spec.label,
        "key": spec.key,
        "corr_z_vs_ue_within_ua": corr_z_ue,
        "corr_err_vs_ua_within_ue": corr_err_ua,
    })

df_unique = pd.DataFrame(rows)
EXPORTER.export_table(df_unique, "unique_information", dataset=DATASET_KEY, split=EVAL_SCOPE)
display(df_unique)



In [ ]:
# Phase 8H: DIDO synergy (Ua vs DIDO vs blend)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

USE_CALIBRATED_SIGMAS_LOCAL = USE_CALIBRATED_SIGMAS if "USE_CALIBRATED_SIGMAS" in globals() else True
EVAL_SCOPE_LOCAL = "id_test"  # "id_test" | "id_test+ood"
OOD_LABELS_LOCAL = list(OOD_LABELS) if "OOD_LABELS" in globals() else []
WEIGHTS = np.linspace(0.0, 1.0, 21)

DIDO_SIGNAL = settings.get("dido_signal", "dido_strength_raw")
DIDO_CANDIDATES = ["dido_entropy_raw", "dido_vacuity_raw", "dido_strength_raw"]
DIDO_ALLOW_FLIP = True

try:
    TAU = float(TAU)
except Exception as e:
    raise RuntimeError("TAU not defined; run the large-error cell first.") from e


def _gains(score: np.ndarray, y_large: np.ndarray):
    d = pd.DataFrame({"score": score, "y": y_large}).dropna()
    if d.empty or d["y"].nunique() < 2:
        return None
    d = d.sort_values("score", ascending=False, kind="mergesort")
    total_pos = float(d["y"].sum())
    if total_pos == 0:
        return None
    cum_pos = d["y"].cumsum().to_numpy(float)
    recall = cum_pos / total_pos
    coverage = np.arange(1, len(d) + 1) / float(len(d))
    aurc = float(np.trapezoid(recall, coverage))
    return coverage, recall, aurc


def _load_for_scope(spec: MethodSpec) -> pd.DataFrame:
    parts = []
    df_id = method_id_frame(spec, "test").copy()
    df_id["dataset"] = "id:test"
    parts.append(df_id)

    if EVAL_SCOPE_LOCAL.strip().lower() == "id_test+ood":
        for ood_label in OOD_LABELS_LOCAL:
            try:
                df_o = method_ood_frame(spec, ood_label).copy()
                df_o["dataset"] = f"ood:{ood_label}"
                parts.append(df_o)
            except Exception as e:
                print(f"[skip] {spec.label} / ood={ood_label}: {type(e).__name__}: {e}")

    return pd.concat(parts, ignore_index=True)


base_spec = next((m for m in METHODS if m.key == "lpl_mc"), None)
if base_spec is None:
    print("[dido synergy] Laplace + MC not available; skipping.")
else:
    dido_spec = next((m for m in METHODS if m.kind == "dido" and m.key == "dido_lpl"), None)
    if dido_spec is None:
        print("[dido synergy] DIDO Laplace not available; skipping.")
    else:
        try:
            df_d = _load_for_scope(dido_spec)
        except Exception as e:
            print(f"[dido synergy] failed to load DIDO: {type(e).__name__}: {e}")
            df_d = None

        if df_d is not None:
            if str(DIDO_SIGNAL).lower() == "auto":
                dido_col = next((c for c in DIDO_CANDIDATES if c in df_d.columns), None)
            else:
                dido_col = DIDO_SIGNAL if DIDO_SIGNAL in df_d.columns else None
            if dido_col is None:
                print(f"[dido synergy] DIDO signal not found: {DIDO_SIGNAL!r}")
            else:
                df_pred = _load_for_scope(base_spec)
                if "id" not in df_d.columns or "id" not in df_pred.columns:
                    print("[dido synergy] Missing 'id' for joining; skipping.")
                else:
                    dd = df_d.copy()
                    pp = df_pred[["id", "y_true", "mu", "sigma_ale_raw", "sigma_epi_raw"]].copy()
                    dd["id"] = pd.to_numeric(dd["id"], errors="coerce")
                    pp["id"] = pd.to_numeric(pp["id"], errors="coerce")
                    dd = dd.dropna(subset=["id"])
                    pp = pp.dropna(subset=["id"])
                    dd["id"] = dd["id"].astype(int)
                    pp["id"] = pp["id"].astype(int)

                    merged = dd.merge(pp, on="id", how="inner")
                    if merged.empty:
                        print("[dido synergy] No overlap between DIDO and base preds.")
                    else:
                        y = merged["y_true"].to_numpy(float)
                        mu = merged["mu"].to_numpy(float)
                        abs_err = np.abs(y - mu)
                        yL = (abs_err > TAU).astype(int)

                        s_a_raw = merged.get("sigma_ale_raw", pd.Series(np.nan, index=merged.index)).to_numpy(float)
                        s_e_raw = merged.get("sigma_epi_raw", pd.Series(np.nan, index=merged.index)).fillna(0.0).to_numpy(float)
                        if USE_CALIBRATED_SIGMAS_LOCAL:
                            s_a, _, _ = calibrated_sigmas(
                                head_type=base_spec.head_type,
                                ale_src=base_spec.aleatoric_source,
                                epi_src=base_spec.epistemic_source,
                                sigma_ale_raw=s_a_raw,
                                sigma_epi_raw=s_e_raw,
                            )
                        else:
                            s_a = np.asarray(s_a_raw, float)

                        dido = pd.to_numeric(merged[dido_col], errors="coerce").to_numpy(float)
                        if DIDO_ALLOW_FLIP:
                            g_pos = _gains(dido, yL)
                            g_neg = _gains(-dido, yL)
                            if g_neg is not None and (g_pos is None or g_neg[2] > g_pos[2]):
                                dido = -dido
                                dido_col = dido_col + " (flipped)"

                        g_ua = _gains(s_a, yL)
                        g_dido = _gains(dido, yL)

                        best = {"w": None, "aurc": -np.inf, "g": None}
                        for w in WEIGHTS:
                            score = (1.0 - w) * s_a + w * dido
                            g = _gains(score, yL)
                            if g is None:
                                continue
                            if g[2] > best["aurc"]:
                                best = {"w": float(w), "aurc": float(g[2]), "g": g}

                        synergy = float(best["aurc"] - max(g_ua[2], g_dido[2])) if best["g"] is not None else float("nan")

                        plt.figure(figsize=(7.5, 5))
                        plt.plot([0, 1], [0, 1], "k--", alpha=0.4, label="random")
                        plt.plot(g_ua[0], g_ua[1], "--", lw=2, label=f"Ua (AURC={g_ua[2]:.3f})")
                        plt.plot(g_dido[0], g_dido[1], ":", lw=2, label=f"DIDO (AURC={g_dido[2]:.3f})")
                        if best["g"] is not None:
                            plt.plot(best["g"][0], best["g"][1], "-", lw=2, label=f"BLEND w={best['w']:.2f} (AURC={best['aurc']:.3f})")
                        plt.xlabel("Coverage reviewed")
                        plt.ylabel("Recall of large errors")
                        plt.title(f"Large-error gains: Ua vs DIDO vs blend (scope={EVAL_SCOPE_LOCAL}, tau={TAU:.3g}, synergy={synergy:+.3f})")
                        plt.ylim(0, 1.05)
                        plt.grid(True, alpha=0.2)
                        plt.legend(frameon=False)
                        plt.tight_layout()
                        savefig(plt.gcf(), f"dido_synergy_{EVAL_SCOPE_LOCAL}_{'cal' if USE_CALIBRATED_SIGMAS_LOCAL else 'raw'}", dataset=DATASET_KEY, split=EVAL_SCOPE_LOCAL)
                        plt.show()


## Phase 9: NF vs analytic + cohort diagnostics

Compare NF vs analytic heads and diagnose cohort coverage differences.

In [ ]:
# Phase 9A: NF vs analytic (Laplace + Gauss) - CRPS + PICP/PIW + cohort noise stats

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

USE_CALIBRATED_SIGMAS_LOCAL = USE_CALIBRATED_SIGMAS if "USE_CALIBRATED_SIGMAS" in globals() else True
PRED_COVER = 0.90

if "gaussian_crps_mean" not in globals():
    from scipy.stats import norm as _norm
    def gaussian_crps_mean(y: np.ndarray, mu: np.ndarray, sigma: np.ndarray) -> float:
        y = np.asarray(y, float)
        mu = np.asarray(mu, float)
        sigma = np.asarray(sigma, float)
        mask = np.isfinite(y) & np.isfinite(mu) & np.isfinite(sigma) & (sigma > 0)
        if not np.any(mask):
            return float("nan")
        y = y[mask]
        mu = mu[mask]
        sigma = sigma[mask]
        z = (y - mu) / sigma
        crps = sigma * (z * (2 * _norm.cdf(z) - 1) + 2 * _norm.pdf(z) - 1 / np.sqrt(np.pi))
        return float(np.mean(crps))


def _find_spec(head_type: str, kind: str, epi_src: str | None = None) -> MethodSpec | None:
    for s in METHODS:
        if s.head_type != head_type or s.kind != kind:
            continue
        if epi_src is None:
            return s
        if str(s.epistemic_source).lower() == epi_src:
            return s
    return None


def _sigma_total(spec: MethodSpec, df: pd.DataFrame) -> np.ndarray:
    s_a = pd.to_numeric(df.get("sigma_ale_raw"), errors="coerce").to_numpy(float)
    s_e = pd.to_numeric(df.get("sigma_epi_raw"), errors="coerce").fillna(0.0).to_numpy(float)
    if USE_CALIBRATED_SIGMAS_LOCAL:
        _, _, s_t = calibrated_sigmas(
            head_type=spec.head_type,
            ale_src=spec.aleatoric_source,
            epi_src=spec.epistemic_source,
            sigma_ale_raw=s_a,
            sigma_epi_raw=s_e,
        )
    else:
        s_t = np.sqrt(np.maximum(np.nan_to_num(s_a, nan=0.0) ** 2 + np.nan_to_num(s_e, nan=0.0) ** 2, 0.0))
    return np.asarray(s_t, float)


specs = {
    "laplace_analytic": _find_spec("laplace", "regression", "none"),
    "laplace_nf": _find_spec("laplace", "nf"),
    "gauss_analytic": _find_spec("gauss", "regression", "none"),
    "gauss_nf": _find_spec("gauss", "nf"),
}

rows = []
for name, spec in specs.items():
    if spec is None:
        continue
    try:
        df = method_id_frame(spec, "test")
    except Exception as e:
        print(f"[skip NF/analytic] {name}: {type(e).__name__}: {e}")
        continue
    if "y_true" not in df.columns or "mu" not in df.columns:
        continue

    y = pd.to_numeric(df["y_true"], errors="coerce").to_numpy(float)
    mu = pd.to_numeric(df["mu"], errors="coerce").to_numpy(float)
    mae = float(np.nanmean(np.abs(y - mu)))
    s_t = _sigma_total(spec, df)
    crps = gaussian_crps_mean(y, mu, s_t)

    z = float(norm.ppf((1.0 + PRED_COVER) / 2.0))
    lower = mu - z * s_t
    upper = mu + z * s_t
    picp = float(np.nanmean((y >= lower) & (y <= upper)))
    piw = float(np.nanmean(upper - lower))

    rows.append({
        "method": spec.label,
        "key": spec.key,
        "group": name,
        "MAE": mae,
        "CRPS": crps,
        "PICP90": picp,
        "PIW90": piw,
        "n": int(len(df)),
    })

df_nf_cmp = pd.DataFrame(rows).sort_values(["group", "MAE"])
EXPORTER.export_table(df_nf_cmp, "nf_vs_analytic_summary", dataset=DATASET_KEY, split="test")
display(df_nf_cmp)

# Simple barh for CRPS and PICP/PIW
if not df_nf_cmp.empty:
    for metric in ["CRPS", "PICP90", "PIW90"]:
        sub = df_nf_cmp[np.isfinite(df_nf_cmp[metric].to_numpy(float))].copy()
        if sub.empty:
            continue
        sub = sub.sort_values(metric, ascending=True)
        plt.figure(figsize=(8, max(3.5, 0.5 * len(sub))))
        y = np.arange(len(sub))
        vals = sub[metric].to_numpy(float)
        plt.barh(y, vals, alpha=0.85)
        plt.yticks(y, sub["method"].tolist())
        plt.gca().invert_yaxis()
        plt.xlabel(metric)
        plt.title(f"NF vs analytic: {metric} (ID:test)")
        plt.grid(True, axis="x", alpha=0.2)
        plt.gca().ticklabel_format(style="plain", axis="x", useOffset=False)
        xmax = float(vals.max()) if len(vals) else 0.0
        pad = 0.02 * xmax
        for yi, v in zip(y, vals):
            plt.text(max(v - pad, 0.0), yi, f"{v:.2f}", va="center", ha="right", fontsize=9, color="black")
        plt.tight_layout()
        savefig(plt.gcf(), f"nf_vs_analytic_{metric.lower()}", dataset=DATASET_KEY, split="test")
        plt.show()

# Cohort noise stats (aleatoric vs abs error) for these methods
_coh = COHORTS if isinstance(COHORTS, dict) else {}
dataset_csv = _resolve_path(_coh.get("dataset_csv"))
if not _coh.get("enabled", False) or not dataset_csv or not dataset_csv.exists():
    print("[cohorts] disabled or dataset_csv missing; skipping noise stats.")
else:
    id_col = _coh.get("id_col", "id")
    cohort_defs = _coh.get("cohort_defs", {}) or {}
    age_col = _coh.get("age_col")
    age_bins = _coh.get("age_bins")
    age_from_cols = _coh.get("age_from_cols", []) or []
    min_n = int(_coh.get("min_cohort_n", 100))

    usecols = {id_col}
    for cols in cohort_defs.values():
        for c in cols:
            if c != "age_bucket":
                usecols.add(c)
    if age_col and age_col != "age_bucket" and not age_from_cols:
        usecols.add(age_col)
    for c in age_from_cols:
        usecols.add(c)

    header_cols = pd.read_csv(dataset_csv, nrows=0).columns.tolist()
    usecols = [c for c in usecols if c in header_cols]

    df_meta = pd.read_csv(dataset_csv, usecols=usecols)

    if age_col and age_col not in df_meta.columns and len(age_from_cols) == 2:
        a0, a1 = age_from_cols
        if a0 in df_meta.columns and a1 in df_meta.columns:
            df_meta[age_col] = pd.to_numeric(df_meta[a0], errors="coerce") - pd.to_numeric(df_meta[a1], errors="coerce")

    if age_col and age_bins and age_col in df_meta.columns:
        df_meta[age_col] = pd.to_numeric(df_meta[age_col], errors="coerce")
        labels = [f"{age_bins[i]}-{age_bins[i+1]}" for i in range(len(age_bins) - 1)]
        df_meta["age_bucket"] = pd.cut(df_meta[age_col], bins=age_bins, labels=labels, include_lowest=True)

    if id_col != "id" and id_col in df_meta.columns:
        df_meta = df_meta.rename(columns={id_col: "id"})
    if "id" in df_meta.columns:
        df_meta["id"] = pd.to_numeric(df_meta["id"], errors="coerce")
        df_meta = df_meta.dropna(subset=["id"]).drop_duplicates(subset=["id"])

    noise_rows = []
    for name, spec in specs.items():
        if spec is None:
            continue
        try:
            df = method_id_frame(spec, "test")
        except Exception as e:
            print(f"[skip noise] {name}: {type(e).__name__}: {e}")
            continue
        if "y_true" not in df.columns or "mu" not in df.columns:
            continue
        y = pd.to_numeric(df["y_true"], errors="coerce").to_numpy(float)
        mu = pd.to_numeric(df["mu"], errors="coerce").to_numpy(float)
        s_a = pd.to_numeric(df.get("sigma_ale_raw"), errors="coerce").to_numpy(float)
        if USE_CALIBRATED_SIGMAS_LOCAL:
            s_a, _, _ = calibrated_sigmas(
                head_type=spec.head_type,
                ale_src=spec.aleatoric_source,
                epi_src=spec.epistemic_source,
                sigma_ale_raw=s_a,
                sigma_epi_raw=np.zeros_like(s_a),
            )
        df_small = pd.DataFrame({
            "id": pd.to_numeric(df.get("id"), errors="coerce"),
            "abs_err": np.abs(y - mu),
            "sigma_ale": np.asarray(s_a, float),
        }).dropna(subset=["id"])
        df_join = df_small.merge(df_meta, on="id", how="inner")

        for coh_name, cols in cohort_defs.items():
            if any(c not in df_join.columns for c in cols):
                continue
            tmp = df_join.dropna(subset=cols).copy()
            g = tmp.groupby(cols).agg(
                n=("id", "size"),
                mean_abs_err=("abs_err", "mean"),
                mean_sigma_ale=("sigma_ale", "mean"),
            )
            g = g[g["n"] >= min_n]
            if g.empty:
                continue
            corr = float(g["mean_sigma_ale"].corr(g["mean_abs_err"]))
            bias = float((g["mean_sigma_ale"] - g["mean_abs_err"]).mean())
            noise_rows.append({
                "method": spec.label,
                "group": name,
                "cohort": coh_name,
                "n_cohorts": int(len(g)),
                "corr_sigma_ale_vs_err": corr,
                "bias_sigma_ale_minus_err": bias,
            })

    df_noise_nf = pd.DataFrame(noise_rows)
    EXPORTER.export_table(df_noise_nf, "nf_vs_analytic_noise_cohort_stats", dataset=DATASET_KEY, split="test")
    display(df_noise_nf.sort_values(["cohort", "corr_sigma_ale_vs_err"], ascending=[True, False]))


In [ ]:
# Phase 9B: diagnose cohort coverage differences (analytic vs NF)

import numpy as np
import pandas as pd

_coh = COHORTS if isinstance(COHORTS, dict) else {}
dataset_csv = _resolve_path(_coh.get("dataset_csv"))
if not _coh.get("enabled", False) or not dataset_csv or not dataset_csv.exists():
    print("[cohorts] disabled or dataset_csv missing; skipping.")
else:
    id_col = _coh.get("id_col", "id")
    cohort_defs = _coh.get("cohort_defs", {}) or {}
    age_col = _coh.get("age_col")
    age_bins = _coh.get("age_bins")
    age_from_cols = _coh.get("age_from_cols", []) or []
    min_n = int(_coh.get("min_cohort_n", 100))

    usecols = {id_col}
    for cols in cohort_defs.values():
        for c in cols:
            if c != "age_bucket":
                usecols.add(c)
    if age_col and age_col != "age_bucket" and not age_from_cols:
        usecols.add(age_col)
    for c in age_from_cols:
        usecols.add(c)
    usecols = list(usecols)

    # filter usecols to actual CSV columns to avoid ValueError
    csv_cols = list(pd.read_csv(dataset_csv, nrows=0).columns)
    usecols = [c for c in usecols if c in csv_cols]

    df_meta = pd.read_csv(dataset_csv, usecols=usecols)

    if age_col and age_col not in df_meta.columns and len(age_from_cols) == 2:
        a0, a1 = age_from_cols
        if a0 in df_meta.columns and a1 in df_meta.columns:
            df_meta[age_col] = pd.to_numeric(df_meta[a0], errors="coerce") - pd.to_numeric(df_meta[a1], errors="coerce")

    if age_col and age_bins and age_col in df_meta.columns:
        df_meta[age_col] = pd.to_numeric(df_meta[age_col], errors="coerce")
        labels = [f"{age_bins[i]}-{age_bins[i+1]}" for i in range(len(age_bins) - 1)]
        df_meta["age_bucket"] = pd.cut(df_meta[age_col], bins=age_bins, labels=labels, include_lowest=True)

    if id_col != "id" and id_col in df_meta.columns:
        df_meta = df_meta.rename(columns={id_col: "id"})
    if "id" in df_meta.columns:
        df_meta["id"] = pd.to_numeric(df_meta["id"], errors="coerce")
        df_meta = df_meta.dropna(subset=["id"]).drop_duplicates(subset=["id"])

    compare = {
        "laplace_analytic": _find_spec("laplace", "regression", "none"),
        "laplace_nf": _find_spec("laplace", "nf"),
    }

    rows = []
    for name, spec in compare.items():
        if spec is None:
            continue
        try:
            df = method_id_frame(spec, "test")
        except Exception as e:
            print(f"[skip diag] {name}: {type(e).__name__}: {e}")
            continue
        df_ids = pd.to_numeric(df.get("id"), errors="coerce").dropna()
        df_small = pd.DataFrame({"id": df_ids})

        joined = df_small.merge(df_meta, on="id", how="inner")

        for coh_name, cols in cohort_defs.items():
            if any(c not in joined.columns for c in cols):
                continue
            tmp = joined.dropna(subset=cols).copy()
            key = tmp[cols].astype(str).agg(" / ".join, axis=1)
            counts = key.value_counts()
            rows.append({
                "method": spec.label,
                "group": name,
                "cohort": coh_name,
                "eval_rows": int(len(df)),
                "rows_with_id": int(len(df_small)),
                "rows_joined": int(len(joined)),
                "n_cohorts": int(counts.size),
                "n_cohorts_ge_min_n": int((counts >= min_n).sum()),
            })

    df_diag = pd.DataFrame(rows)
    EXPORTER.export_table(df_diag, "cohort_join_diagnostics", dataset=DATASET_KEY, split="test")
    display(df_diag.sort_values(["cohort", "group"]))


In [ ]:
# Write export manifest.
run_dirs = None
if "RUN_DIRS" in globals():
    run_dirs = {k: (str(v) if v else None) for k, v in RUN_DIRS.items()}

extra = {
    "config_path": str(CONFIG_PATH) if "CONFIG_PATH" in globals() else None,
    "eval_root": str(EVAL_ROOT) if "EVAL_ROOT" in globals() else None,
    "dataset_key": DATASET_KEY if "DATASET_KEY" in globals() else None,
    "ood_labels": OOD_LABELS if "OOD_LABELS" in globals() else None,
    "large_error": LARGE_ERROR if "LARGE_ERROR" in globals() else None,
    "tau": float(TAU) if "TAU" in globals() else None,
    "run_dirs": run_dirs,
}
EXPORTER.write_manifest(extra=extra, project_root=PROJECT_ROOT)
